修正版

In [ ]:
"""競技かるた: VoskによるWAV/マイク音声認識と札予測。

必要: pip install vosk numpy sounddevice matplotlib ipywidgets
モデル: ./model_standard または ./model_small
辞書: ./data/karuta_dictionary.csv (18列の新形式)
"""
from __future__ import annotations

import csv
import math
import glob
import json
import os
import random
import re
import sys
import time
import unicodedata
import wave
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime

try:
    from IPython.display import HTML, display, update_display, clear_output
    from IPython import get_ipython
except ImportError:
    HTML = display = update_display = clear_output = get_ipython = None
from pathlib import Path

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

import numpy as np
from vosk import KaldiRecognizer, Model, SetLogLevel

try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    # Jupyter Notebook / IPython では __file__ が定義されない
    ROOT = Path.cwd()
SAMPLE_RATE = 16000
CHUNK_SIZE = 1600
VOLUME_THRESHOLD = 0.01
WAIT_SECONDS = 0.8  # 下の句の終了判定に必要な連続無音時間
UPPER_END_SILENCE_SECONDS = 1.2  # 上の句の読み終わり判定に必要な連続無音時間
UPPER_MIN_TOTAL_SECONDS = 3.5  # 上の句の開始から、読み終わりと判断しない最短時間
# 読手によって長い間を取る場合は、この2値を調整してください。
DEFAULT_LIMIT_SECONDS = 8.0  # 初語が見つからないときの安全終了時間
TIME_CORRECTION_SECONDS = 0.5  # 最短認識時間への加算値（実行時に変更可）
RESULTS_CSV = ROOT / 'results' / 'batch_recognition_history.csv'
RECORD_DIR = ROOT / 'test_recordings'
PLOT_DIR = ROOT / 'rms_plots'
AUDIO_DIR = ROOT / 'recordings'  # マイク録音とWAV解析元音声の保存先
# 詳細CSVはresultsとは別の場所に、実行ごとに保存。
DETAIL_DIR = ROOT / 'recognition_details'
RUN_ID = None
RUN_DETAIL_DIR = None
RUN_PLOT_DIR = None
RUN_AUDIO_DIR = None
RUN_DIVIDER_WRITTEN = False


def start_run():
    """main()を実行するたびに、実行IDと詳細保存フォルダを作成する。"""
    global RUN_ID, RUN_DETAIL_DIR, RUN_PLOT_DIR, RUN_AUDIO_DIR, RUN_DIVIDER_WRITTEN
    RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    RUN_DETAIL_DIR = DETAIL_DIR / RUN_ID
    RUN_DETAIL_DIR.mkdir(parents=True, exist_ok=True)
    RUN_PLOT_DIR = PLOT_DIR / RUN_ID
    RUN_PLOT_DIR.mkdir(parents=True, exist_ok=True)
    RUN_AUDIO_DIR = AUDIO_DIR / RUN_ID
    RUN_AUDIO_DIR.mkdir(parents=True, exist_ok=True)
    RUN_DIVIDER_WRITTEN = False
    print(f'[今回の実行ID] {RUN_ID}')
    print(f'[詳細CSV保存先] {RUN_DETAIL_DIR}')
    print(f'[RMS画像保存先] {RUN_PLOT_DIR}')
    print(f'[録音WAV保存先] {RUN_AUDIO_DIR}')
    return RUN_ID


DICTIONARIES = {
    '1': ('ひらがな版（統一文字数辞書）', ROOT / 'data/karuta_hiragana_aligned.csv', 'aligned_hiragana'),
    '2': ('カタカナ版（統一文字数辞書）', ROOT / 'data/karuta_katakana_aligned.csv', 'aligned_katakana'),
    '3': ('決まり字・手動追加版（サマリー辞書）', ROOT / 'data/karuta_card_summary_by_kimari_len.csv', 'summary'),
    '4': ('VOSK専用辞書', ROOT / 'data/karuta_vosk_dic.csv', 'vosk_custom'),
    '5': ('VOSK特化辞書（旧形式）', ROOT / 'data/dic_VOSK_cus_converted.csv', 'vosk_cus_converted'),
    '6': ('カスタムテキスト入力（動的辞書）', None, 'custom_input'),
    '7': ('新・かるた統合辞書', ROOT / 'data/karuta_dictionary.csv', 'new_karuta'),
}

# 新辞書の列名。基本は「決まり字 + 分割パーツ + 手動追加 + VOSK特化」。
NEW_COLUMNS = [
    'カタカナ決まり字', 'ひらがな決まり字',
    'カタカナ決まり字パーツ', 'ひらがな決まり字パーツ',
    'カタカナパーツ(1句目)', 'ひらがなパーツ(1句目)',
    'カタカナパーツ(2句目)', 'ひらがなパーツ(2句目)',
    '手動追加パーツ', '誤識別追加',
    'VOSK特化辞書カタカナ', 'VOSK特化辞書ひらがな',
]
# ===== 3種類の重み付けを独立してON/OFFできます =====
# Falseにするとその方式は無効。すべてFalseなら全語1.0倍です。
USE_COLUMN_WEIGHTS = True        # CSVの列ごと
USE_WORD_WEIGHTS = True          # 全札共通・単語ごと
USE_CARD_WORD_WEIGHTS = True    # 特定の札・単語ごと

# ① 列ごとの重み（初期値はすべて1.0）
COLUMN_WEIGHTS = {col: 1.0 for col in NEW_COLUMNS}
COLUMN_WEIGHTS["カタカナ決まり字"] = 1.5
COLUMN_WEIGHTS["ひらがな決まり字"] = 1.5
COLUMN_WEIGHTS["手動追加パーツ"] = 1.0

# ② 単語ごとの重み（札を問わず適用）
WORD_WEIGHTS = {
    '伊佐': 1.5,
    'アキノ': 1.0,
    'カモ': 1.6,
}

# ③ 特定の札＋単語ごとの重み（現在はON）
CARD_WORD_WEIGHTS = {
    ('004', 'ウラ'): 1.6,
    ('028', 'ヤマ'): 1.3,
    ('032', 'ヤマ'): 1.5,
    ('035', 'ヒト'): 1.5,
    ('099', 'ヒト'): 1.6,
    ('056', 'アラ'): 1.6,
    ('063', 'ワタ'): 0.8,
    ('081', 'オト'): 0.8,
}

# 優先順位：③（札＋単語）> ②（単語）> ①（列）> 1.0
# 3方式をすべてONにした場合も、重みを掛け算しません。
# カタカナ/ひらがな等の表記はnormalize()によって統一して検索します。
# 全ての重みには0以上の有限数を指定してください。

def make_weight_settings():
    """設定の変更を辞書再読込時に反映し、指定語を正規化して検索できるようにする。"""
    def checked_weight(value, where):
        value = float(value)
        if not math.isfinite(value) or value < 0:
            raise ValueError(f'{where}: 重みは0以上の有限数で指定してください: {value}')
        return value

    by_word = {normalize(w): checked_weight(weight, f'WORD_WEIGHTS[{w!r}]')
               for w, weight in WORD_WEIGHTS.items()}
    by_card = {(card_id(cid), normalize(w)): checked_weight(weight, f'CARD_WORD_WEIGHTS[{cid!r}, {w!r}]')
               for (cid, w), weight in CARD_WORD_WEIGHTS.items()}
    by_column = {col: checked_weight(weight, f'COLUMN_WEIGHTS[{col!r}]')
                 for col, weight in COLUMN_WEIGHTS.items()}
    return by_column, by_word, by_card

# ===== 漢字の加点用文字数（ここをON/OFF） =====
USE_HIRAGANA_LENGTH_FOR_KANJI = True  # True: 漢字を読み（ひらがな）に換算した長さで加点

# 自動変換が間違う語はここに正しい読みを記入してください（例：固有名詞）。
KANJI_READING_OVERRIDES = {
    '秋の': 'あきの',
    '秋野': 'あきの',
    '明乃': 'あきの',
    '秋風': 'あきかぜ',
    '月見': 'つきみ',
}

_kakasi_instance = None


def hiragana_reading(word: str) -> str:
    """漢字を含む単語のみ、読みをひらがなにする。変換できなければ元の表記を使う。"""
    global _kakasi_instance
    import unicodedata
    w = str(word)
    if w in KANJI_READING_OVERRIDES:
        return KANJI_READING_OVERRIDES[w]
    if not any('\u3400' <= c <= '\u9fff' or c == '々' for c in w):
        # ひらがな/カタカナは通常の文字数を使う
        return w
    if _kakasi_instance is None:
        try:
            from pykakasi import kakasi
        except ImportError as exc:
            raise ImportError('漢字のひらがな換算をONにするには、Notebookで %pip install pykakasi を実行してください。') from exc
        _kakasi_instance = kakasi()
    try:
        return ''.join(part['hira'] for part in _kakasi_instance.convert(w)) or w
    except Exception:
        return w


def score_length(word: str) -> int:
    """認識語1つの加点基準文字数。漢字換算OFFならこれまでどおり。"""
    if USE_HIRAGANA_LENGTH_FOR_KANJI:
        return len(hiragana_reading(word))
    return len(word)


def normalize(value: object) -> str:
    """Voskの空白・表記揺れによる簡単な取りこぼしを減らす。"""
    return re.sub(r'\s+', '', unicodedata.normalize('NFKC', str(value)).strip()).lower()


def card_id(value: object) -> str:
    s = str(value).strip()
    try:
        return f'{int(s):03d}'
    except ValueError:
        return s


def words_in_cell(value: object) -> list[str]:
    return [normalize(x) for x in str(value or '').replace('　', ' ').split() if normalize(x)]


@dataclass
class DictionaryData:
    name: str
    path: str
    words_by_card: dict[str, dict[str, float]]
    grammar: list[str]
    durations: dict[str, float]  # 札ごとの最短認識時間（最大時間ではない）


def load_dictionary(choice: str, custom_id: str = 'CUSTOM', new_mode: str = '1') -> DictionaryData:
    name, path, kind = DICTIONARIES[choice]
    words_by_card: dict[str, dict[str, float]] = defaultdict(dict)
    durations: dict[str, float] = {}  # 札ごとの最短認識時間
    column_weights, word_weights, card_word_weights = make_weight_settings()

    def add(cid: str, word: str, column: str | None = None):
        word = normalize(word)
        if not word:
            return
        # 基本重みは列設定から取得。同一札内で複数列にあれば最大値。
        base_weight = (column_weights.get(column, 1.0)
                       if USE_COLUMN_WEIGHTS and column else 1.0)
        if USE_WORD_WEIGHTS and word in word_weights:
            base_weight = word_weights[word]
        if USE_CARD_WORD_WEIGHTS and (cid, word) in card_word_weights:
            base_weight = card_word_weights[(cid, word)]
        # 重複除外。列単位の重みが異なる場合は大きいほうを採用。
        # 単語／札＋単語の指定がある場合は同じ値なのでそのまま。
        words_by_card[cid][word] = max(words_by_card[cid].get(word, 0.0), base_weight)

    if kind == 'custom_input':
        raw = input('【単語入力】スペース区切り (例: はるす はる ハルス) -> ')
        for word in words_in_cell(raw):
            add(custom_id, word)
    elif kind == 'new_karuta':
        if not path.exists():
            raise FileNotFoundError(f'辞書がありません: {path}')
        with path.open('r', encoding='utf-8-sig', newline='') as f:
            reader = csv.DictReader(f)
            headers = [h.strip() for h in (reader.fieldnames or [])]
            if '札番号' not in headers:
                raise ValueError('新辞書に「札番号」列がありません。CSVの先頭行を確認してください。')
            allowed = (['VOSK特化辞書カタカナ', 'VOSK特化辞書ひらがな']
                       if new_mode == '2' else NEW_COLUMNS)
            existing = [col for col in allowed if col in headers]
            if not existing:
                raise ValueError('新辞書に利用できる語彙列がありません。列名を確認してください。')
            for raw_row in reader:
                row = {str(k).strip(): v for k, v in raw_row.items() if k is not None}
                cid = card_id(row.get('札番号', ''))
                if not cid or cid in ('000', '00'):
                    continue
                try:
                    durations[cid] = max(0.1, float(row.get('認識時間', '') or DEFAULT_LIMIT_SECONDS))
                except (TypeError, ValueError):
                    pass
                for col in existing:
                    for w in words_in_cell(row.get(col, '')):
                        add(cid, w, column=col)
        print('読み込み列:', ' / '.join(existing))
    else:
        if path is None or not path.exists():
            raise FileNotFoundError(f'辞書がありません: {path}')
        with path.open('r', encoding='utf-8-sig', newline='') as f:
            reader = csv.reader(f)
            next(reader, None)
            for row in reader:
                if not row or row[0].strip() in ('0', '00', ''):
                    continue
                cid = card_id(row[0])
                if kind == 'vosk_cus_converted':
                    cells = row[2:4]
                elif kind == 'vosk_custom':
                    cells = row[3:]
                    if len(row) > 2:
                        try:
                            durations[cid] = max(0.1, float(row[2]))
                        except ValueError:
                            pass
                elif kind in ('aligned_hiragana', 'aligned_katakana'):
                    cells = row[5:6]
                else:
                    cells = row[7:14]
                for cell in cells:
                    for w in words_in_cell(cell):
                        add(cid, w)

    words_by_card = {cid: words for cid, words in words_by_card.items() if words}
    if not words_by_card:
        raise ValueError('辞書に認識できる語彙がありません。')
    grammar = sorted(set(w for words in words_by_card.values() for w in words), key=lambda x: (-len(x), x))
    print(f'[辞書ロード完了] {name}: {len(words_by_card)}札 / {len(grammar)}語 / {sum(map(len, grammar))}文字（全札の重複を除外）')
    print(f'[認識時間] CSVの数値は札ごとの最短認識時間です（指定札数: {len(durations)}）。')
    print(f'[重み付け] 列={USE_COLUMN_WEIGHTS} / 単語={USE_WORD_WEIGHTS} / 札＋単語={USE_CARD_WORD_WEIGHTS}')
    return DictionaryData(name, str(path) if path else '手動入力', words_by_card, grammar, durations)


def export_dictionary_csv(dic: DictionaryData, output: Path | None = None) -> Path:
    """札単位で重複を排除し、辞書の登録内容を1枚1行のCSVに保存する。"""
    output = output or ROOT / 'results' / 'karuta_dictionary_preview.csv'
    output.parent.mkdir(parents=True, exist_ok=True)
    try:
        f = output.open('w', encoding='utf-8-sig', newline='')
    except PermissionError:
        output = output.with_name(output.stem + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f') + output.suffix)
        f = output.open('w', encoding='utf-8-sig', newline='')
        print('[注意] 元のCSVが使用中のため、別名で保存します。')
    with f:
        writer = csv.writer(f)
        writer.writerow(['札番号', '最短認識時間(秒)', '重複なし単語数',
                         '重複なし総文字数', '加点用総文字数', '平均単語長', '最短単語長', '最長単語長', '登録単語一覧', '単語別加点用文字数', '単語別重み', '加重満点スコア'])
        for cid in sorted(dic.words_by_card):
            words = sorted(dic.words_by_card[cid], key=lambda w: (-len(w), w))
            lengths = [len(w) for w in words]
            writer.writerow([cid, dic.durations.get(cid, ''), len(words), sum(lengths),
                             sum(score_length(w) for w in words),
                             round(sum(lengths) / len(words), 2) if words else 0,
                             min(lengths, default=0), max(lengths, default=0), ' '.join(words),
                             ' '.join(f'{w}:{score_length(w)}' for w in words),
                             ' '.join(f'{w}:{dic.words_by_card[cid][w]:g}' for w in words),
                             round(sum(score_length(w) * dic.words_by_card[cid][w] for w in words), 4)])
    print(f'[辞書内容CSV保存] {output.resolve()}')
    return output

In [2]:
class CardPredictor:
    """部分認識を毎回再計算するため、同じ語で点数が二重加算されない。"""
    def __init__(self, dictionary: DictionaryData):
        self.dictionary = dictionary
        self.index: dict[str, list[tuple[str, float]]] = defaultdict(list)
        self.max_scores = {}
        for cid, words in dictionary.words_by_card.items():
            self.max_scores[cid] = sum(score_length(w) * weight for w, weight in words.items()) or 1.0
            for word, weight in words.items():
                self.index[word].append((cid, weight))
        self.all_words = sorted(self.index, key=lambda x: (-len(x), x))

    def predict(self, transcript: str) -> tuple[list[tuple[str, float, float]], list[str]]:
        compact = normalize(transcript)
        if not compact:
            return [], []
        # まず語単位の一致。それ以外に、Voskが単語を連結した出力も拾う。
        tokens = set(words_in_cell(transcript))
        matched = set(w for w in tokens if w in self.index)
        covered = [False] * len(compact)
        for w in sorted(matched, key=len, reverse=True):
            at = compact.find(w)
            while at >= 0:
                for i in range(at, at + len(w)):
                    covered[i] = True
                at = compact.find(w, at + 1)
        for w in self.all_words:
            if w in matched or len(w) < 2:
                continue
            at = compact.find(w)
            while at >= 0:
                if not any(covered[at:at + len(w)]):
                    matched.add(w)
                    for i in range(at, at + len(w)):
                        covered[i] = True
                    break
                at = compact.find(w, at + 1)

        scores = defaultdict(float)
        for w in matched:
            for cid, weight in self.index[w]:
                scores[cid] += score_length(w) * weight
        ranking = sorted(((cid, min(100.0, 100.0 * score / self.max_scores[cid]), score)
                          for cid, score in scores.items()), key=lambda x: (-x[1], -x[2], x[0]))
        return ranking, sorted(matched, key=lambda x: (-len(x), x))


def reader_prompt() -> tuple[str, str]:
    choice = input('【読手】1: 稲葉専任 / 2: 芹野専任 / 3: その他 -> ').strip()
    if choice == '1':
        return '稲葉専任読手', '男性'
    if choice == '2':
        return '芹野専任読手', '女性'
    name = input(' -> 読手名を入力: ').strip() or '未指定読手'
    gender_choice = input(' -> 性別 1:男性 / 2:女性 / 3:未指定 -> ').strip()
    return name, {'1': '男性', '2': '女性'}.get(gender_choice, '未指定')


def input_number(prompt: str, default: float) -> float:
    raw = input(f'{prompt} [Enter: {default}] -> ').strip()
    if not raw:
        return default
    try:
        return max(0.1, float(raw))
    except ValueError:
        print('数値ではないため規定値を使用します。')
        return default


def create_recognizer(model: Model, dictionary: DictionaryData) -> KaldiRecognizer:
    grammar = dictionary.grammar + ['[unk]']
    rec = KaldiRecognizer(model, SAMPLE_RATE, json.dumps(grammar, ensure_ascii=False))
    return rec


def safe_vosk_text(result: str, key: str = 'text') -> str:
    try:
        return str(json.loads(result).get(key, '') or '')
    except (ValueError, TypeError):
        return ''

In [8]:
class NotebookStatus:
    """Notebookで高さを抑えた固定ダッシュボードを表示・上書きする。"""
    def __init__(self, enabled: bool = True):
        self.enabled = bool(enabled and display is not None)
        self.widget = widgets.HTML(value='') if self.enabled and widgets is not None else None
        self.created = False
        self.display_id = 'karuta_dashboard_' + datetime.now().strftime('%H%M%S_%f')

    def show(self, session, message: str = ''):
        if not self.enabled:
            return
        from html import escape
        def h(value):
            return escape(str(value))

        elapsed = session.elapsed
        target = session.end_at if session.end_at is not None else session.limit
        remaining = max(0.0, target - elapsed)
        phase = {'lower': ('前札の下の句中' if session.voice_seen else '前札の下の句待ち'),
                 'upper_wait': '次の上の句待ち',
                 'upper_active': '認識中',
                 'upper_tail': '上の句の残りを待機中',
                 'lower_wait': '下の句待ち（間奏）',
                 'lower_active': '前札の下の句中',
                 'lower_silence': '下の句終了確認中'}.get(session.phase, session.phase)
        if session.finished and session.phase == 'upper_active':
            phase = '判定完了'
        if message:
            phase += '・' + message

        # 上位5札を固定5行で表示する。棒はCSSで描き、高さを抑える。
        entries = []
        for rank in range(5):
            if rank < len(session.ranking):
                cid, sim, pts = session.ranking[rank]
                maximum = session.predictor.max_scores.get(cid, 1.0)
                pct = max(0, min(100, sim))
                entries.append(
                    f'<tr><td>{rank+1}</td><td><b>{h(cid)}番</b></td>'
                    f'<td>{pts:g}/{maximum:g}</td><td>{sim:.1f}%</td>'
                    f'<td><div style="height:7px;background:#d9e0e6;border-radius:4px;overflow:hidden">'
                    f'<div style="height:100%;width:{pct:.1f}%;background:#16886f"></div></div></td></tr>'
                )
            else:
                entries.append(
                    f'<tr><td>{rank+1}</td><td>---番</td><td>0/---</td><td>0.0%</td>'
                    f'<td><div style="height:7px;background:#d9e0e6;border-radius:4px"></div></td></tr>'
                )

        # 画面には最新3件を1行表示する。全履歴は詳細CSVに保存する。
        logs = getattr(session, 'recognition_logs', [])[-3:]
        log_text = '　｜　'.join(f'{t:.2f}s {h(w)}→{h(card)}番' for t, w, card, _ in logs)
        if not log_text:
            log_text = 'まだ一致語なし'
        transcript = h(session.transcript or '（未認識）')
        first = h(session.first_word or '---')
        ref_card = h(session.first_card or '---')
        timing_state = h('確定' if session.timing_fixed else ('仮設定' if session.first_word else '未設定'))
        timing_candidates = len(session.timing_candidates)
        html = f"""<div style="box-sizing:border-box;max-width:780px;padding:5px 9px;
             border:1px solid #c4cbd3;border-radius:5px;background:#fff;
             color:#1d2939;font:12px/1.25 sans-serif">
          <div style="display:flex;justify-content:space-between;gap:8px;align-items:center;
                      margin:0 0 3px 0;white-space:nowrap">
            <b style="font-size:13px">競技かるた 札予測</b>
            <span>状態：<b>{h(phase)}</b></span>
          </div>
          <div style="padding:3px 5px;background:#f0f3f6;white-space:nowrap;overflow:hidden">
            経過 <b>{elapsed:.2f}s</b>　終了 <b>{target:.2f}s</b>　残り <b>{remaining:.2f}s</b>
            　RMS <b>{session.last_rms:.4f}</b>
          </div>
          <div style="padding:3px 0;white-space:nowrap;overflow:hidden;text-overflow:ellipsis">
            <b>認識：</b>{transcript}
          </div>
          <div style="padding:0 0 3px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;color:#526071">
            初語 {first}　時間基準 {ref_card}番（{timing_state}・候補{timing_candidates}枚）　補正 +{session.correction:.2f}s
          </div>
          <table style="table-layout:fixed;width:100%;border-collapse:collapse;
                        font:12px/1.15 sans-serif;text-align:left">
            <thead><tr style="background:#e9eef3">
              <th style="width:8%">順位</th><th style="width:17%">札</th>
              <th style="width:20%">点/満点</th><th style="width:16%">一致率</th>
              <th style="width:39%">スコア</th>
            </tr></thead>
            <tbody>{''.join(entries)}</tbody>
          </table>
          <div style="margin-top:3px;padding-top:3px;border-top:1px solid #d6dce3;
                      white-space:nowrap;overflow:hidden;text-overflow:ellipsis">
            <b>直近：</b>{log_text}
          </div>
        </div>"""
        # ウィジェット1つだけを更新し、過去の表示を積み重ねない。
        if self.widget is not None:
            self.widget.value = html
            if not self.created:
                display(self.widget)
                self.created = True
        elif not self.created:
            display(HTML(html), display_id=self.display_id)
            self.created = True
        else:
            update_display(HTML(html), display_id=self.display_id)


class RecognitionSession:
    """マイク音声を処理し、Notebook表示と途中経過ログを蓄積する。"""
    def __init__(self, rec: KaldiRecognizer, predictor: CardPredictor, limit: float,
                 start_mode: str, show_partial: bool = True,
                 correction: float = TIME_CORRECTION_SECONDS,
                 notebook_display: bool = False):
        self.rec = rec
        self.predictor = predictor
        self.limit = limit
        self.correction = correction
        self.start_mode = start_mode
        self.show_partial = show_partial
        self.dashboard = NotebookStatus(enabled=notebook_display)
        self.notebook_display = bool(self.dashboard.enabled)
        self.reset()

    @property
    def elapsed(self):
        return (self.total_time - self.upper_start) if self.upper_start is not None else 0.0

    def reset(self):
        self.rec.Reset()
        self.phase = 'lower' if self.start_mode == '2' else 'upper_wait'
        self.total_time = 0.0
        self.last_voice_time = None
        self.voice_seen = False
        self.committed = []
        self.transcript = ''
        self.ranking = []
        self.matched = []
        self.last_display = -1.0
        self.rms_data = []
        self.upper_start = None
        self.finished = False
        self.first_word = None
        self.first_card = None
        self.first_word_at = None
        self.timing_candidates = set()  # 初語に一致する札の候補集合
        self.timing_seen_words = set()  # 時間再設定に使用済みの語
        self.timing_fixed = False  # 1枚まで絞れたときのみ確定
        self.timing_revisions = 0
        self.end_at = None
        self.detail_rows = []
        self.word_markers = []  # (音声先頭からの秒数, 単語)
        self.phase_markers = []  # (音声先頭からの秒数, 状態名)
        self.recognition_logs = []  # (上の句経過秒数, 語, 札番号, 加点)
        self.last_rms = 0.0
        if self.notebook_display:
            self.dashboard.show(self)
        else:
            print('\n[待機] ' + ('前札の下の句を読んでください。無音後に上の句を認識します。'
                                   if self.phase == 'lower' else '上の句を読んでください。'))

    def record_event(self, event: str):
        top = self.ranking[:3]
        row = {
            'イベント': event,
            '全体経過時間(秒)': round(self.total_time, 3),
            '上の句経過時間(秒)': round(self.elapsed, 3),
            '音量RMS': round(self.last_rms, 6),
            '状態': self.phase,
            '認識文字列': self.transcript,
            '一致語': ' '.join(self.matched),
            '初認識語': self.first_word or '',
            '初認識時刻(秒)': round(self.first_word_at, 3) if self.first_word_at is not None else '',
            '基準札番号': self.first_card or '',
            '基準札最短時間(秒)': self.predictor.dictionary.durations.get(self.first_card, '') if self.first_card else '',
            '時間候補札': ' '.join(sorted(self.timing_candidates)),
            '時間基準確定': self.timing_fixed,
            '時間再設定回数': self.timing_revisions,
            '補正値(秒)': self.correction,
            '終了予定(秒)': self.end_at if self.end_at is not None else self.limit,
            '終了まで(秒)': round(max(0.0, (self.end_at if self.end_at is not None else self.limit) - self.elapsed), 3),
        }
        for i in range(3):
            cid, pct, score = top[i] if i < len(top) else ('', '', '')
            row.update({f'第{i+1}候補札': cid, f'第{i+1}類似度(%)': round(pct, 2) if pct != '' else '',
                        f'第{i+1}スコア': round(score, 3) if score != '' else ''})
        self.detail_rows.append(row)

    def show_status(self, message=''):
        if self.notebook_display:
            self.dashboard.show(self, message)
        elif self.show_partial and self.phase == 'upper_active':
            labels = ' / '.join(f'{cid}番 {pct:.1f}%' for cid, pct, _ in self.ranking[:3]) or '候補なし'
            print(f'\r[予測] {labels} | 認識: {self.transcript[:65]}', end=' '*10 + '\r', flush=True)

    def _set_timing_from_candidates(self):
        """候補札の最短時間の最大値を仮採用。1札になれば終了時刻を確定する。"""
        if not self.timing_candidates:
            return
        durations = self.predictor.dictionary.durations
        self.first_card = sorted(self.timing_candidates,
                                 key=lambda cid: (-durations.get(cid, self.limit), cid))[0]
        min_seconds = durations.get(self.first_card, self.limit)
        self.end_at = max(0.0, min_seconds + self.correction)
        self.timing_fixed = len(self.timing_candidates) == 1

    def _update(self, partial: str, force: bool = False, event: str = '部分認識'):
        all_text = ' '.join(self.committed + ([partial] if partial else []))
        if all_text == self.transcript and not force:
            return
        self.transcript = all_text
        self.ranking, self.matched = self.predictor.predict(all_text)
        # 認識された新しい辞書語を、RMSグラフに表示する時刻付きで記録する。
        already = {word for _, word in self.word_markers}
        for word in self.matched:
            if word not in already:
                self.word_markers.append((self.total_time, word))
                if self.ranking:
                    for cid, _, _ in self.ranking:
                        if any(k == cid for k, _ in self.predictor.index.get(word, [])):
                            weight = dict(self.predictor.index[word])[cid]
                            self.recognition_logs.append((self.elapsed, word, cid, score_length(word)*weight))
                            break
        # 時間基準を決めるための語を、音声文字列に現れた順に処理する。
        # 同じ語が部分認識で繰り返されても、再設定・加点は重複させない。
        word_stream = [w for w in words_in_cell(all_text)
                       if w != '[unk]' and w in self.predictor.index]
        compact = normalize(all_text.replace('[unk]', ''))
        if not word_stream and self.matched:
            # 空白なしで連結認識された場合は、重なりのないパーツのみを採用。
            # 「あきの」とその部分文字列「あき」を別々の2語と数えない。
            occupied = set()
            ordered = sorted(self.matched, key=lambda w: (compact.find(w), -len(w), w))
            for w in ordered:
                start = compact.find(w)
                if start < 0 or any(i in occupied for i in range(start, start + len(w))):
                    continue
                word_stream.append(w)
                occupied.update(range(start, start + len(w)))

        for token in word_stream:
            if token in self.timing_seen_words:
                continue
            self.timing_seen_words.add(token)
            word_candidates = {cid for cid, _ in self.predictor.index[token]}
            if self.first_word is None:
                self.first_word = token
                self.first_word_at = self.elapsed
                self.timing_candidates = word_candidates
                self._set_timing_from_candidates()
                self.record_event('初語・時間仮設定' if not self.timing_fixed else '初語・時間確定')
                if not self.notebook_display:
                    print(f'\n[初語] {token}: {len(word_candidates)}札 / '
                          f'基準{self.first_card}番 / 終了予定{self.end_at:.2f}秒')
            elif not self.timing_fixed:
                narrowed = self.timing_candidates & word_candidates
                if narrowed and narrowed != self.timing_candidates:
                    self.timing_candidates = narrowed
                    self.timing_revisions += 1
                    self._set_timing_from_candidates()
                    self.record_event('追加語・時間確定' if self.timing_fixed else '追加語・時間再設定')
                    if not self.notebook_display:
                        print(f'\n[時間再設定] {token}: 候補{len(narrowed)}札 / '
                              f'基準{self.first_card}番 / 終了予定{self.end_at:.2f}秒')
                # 共通候補がない追加語で時間を変更しない（誤認識による揺れを防ぐ）。
        self.record_event(event)
        if self.show_partial and (force or self.total_time - self.last_display >= 0.20):
            self.show_status()
            self.last_display = self.total_time

    def consume(self, data: bytes):
        if self.finished or not data:
            return
        values = np.frombuffer(data, dtype=np.int16).astype(np.float32)
        volume = float(np.sqrt(np.mean(values * values)) / 32768.0) if len(values) else 0.0
        dt = len(values) / SAMPLE_RATE
        self.total_time += dt
        self.last_rms = volume
        self.rms_data.append((self.total_time, volume))
        speaking = volume >= VOLUME_THRESHOLD

        if self.phase == 'lower':
            if speaking:
                self.voice_seen = True
                self.last_voice_time = self.total_time
            elif self.voice_seen and self.last_voice_time is not None and self.total_time - self.last_voice_time >= WAIT_SECONDS:
                self.phase = 'upper_wait'
                self.voice_seen = False
                self.last_voice_time = None
                self.rec.Reset()
                self.record_event('下の句終了')
                if self.notebook_display:
                    self.show_status('上の句待機')
                else:
                    print('\n[切替] 下の句が終了。次の上の句を待機します。')
            return

        if self.phase == 'upper_wait':
            if not speaking:
                return
            self.phase = 'upper_active'
            self.upper_start = self.total_time - dt
            self.voice_seen = True
            self.record_event('認識開始')
            if self.notebook_display:
                self.show_status()
            else:
                print('\n[認識開始]')

        if speaking:
            self.last_voice_time = self.total_time
        if self.rec.AcceptWaveform(data):
            text = safe_vosk_text(self.rec.Result())
            if text:
                self.committed.append(text)
            self._update('', force=True, event='確定認識')
        else:
            self._update(safe_vosk_text(self.rec.PartialResult(), 'partial'))
        # 文字起こしが更新されなくても、経過時間だけは約0.2秒ごとに表示更新する。
        if self.notebook_display and self.total_time - self.last_display >= 0.20:
            self.show_status()
            self.last_display = self.total_time

        if self.end_at is not None:
            if self.elapsed >= self.end_at:
                self.finish()
        elif self.elapsed >= self.limit:
            self.finish()

    def finish(self):
        if self.finished:
            return
        text = safe_vosk_text(self.rec.FinalResult())
        if text:
            self.committed.append(text)
        self._update('', force=True, event='最終文字起こし')
        self.finished = True
        self.record_event('認識終了')
        if self.notebook_display:
            self.show_status('結果確定')
        else:
            print('\n' + '-' * 68)
            if self.ranking:
                print('最終候補: ' + ' / '.join(f'{cid}番 ({pct:.1f}%)' for cid, pct, _ in self.ranking[:5]))
            else:
                print('最終候補: なし（音声・辞書・マイク音量を確認）')
            print(f'認識文字列: {self.transcript or "なし"}')
            print(f'時間基準: 初語={self.first_word or "なし"} / 基準札={self.first_card or "なし"} / 終了時刻={self.end_at if self.end_at is not None else self.limit:.2f}秒 / {"確定" if self.timing_fixed else "仮設定"}')
            print('一致語: ' + (' '.join(self.matched) if self.matched else 'なし'))
            print('-' * 68)

    def result(self):
        winner = self.ranking[0] if self.ranking else ('---', 0.0, 0.0)
        return winner[0], winner[1]

In [9]:
def make_short_output_stem(source: str) -> str:
    """マイク名から「連番_日時」に変換する。WAV入力では元名を利用する。"""
    stem = Path(source).stem
    match = re.fullmatch(r'microphone_(\d{8}_\d{6}_\d{6})_(\d{3})', stem)
    if match:
        stamp, number = match.groups()
        return f'{number}_{stamp}'
    return re.sub(r'[^\w.-]', '_', stem)[:80]


def save_audio_wav(audio_chunks: list[bytes], basename: str, *, partial: bool = False) -> Path | None:
    """マイクから取得したPCM 16-bit/16kHz/モノラル音声をWAVとして保存する。"""
    if not audio_chunks:
        return None
    dest_dir = RUN_AUDIO_DIR if RUN_AUDIO_DIR is not None else AUDIO_DIR
    dest_dir.mkdir(parents=True, exist_ok=True)
    safe_name = make_short_output_stem(basename)
    suffix = '_partial' if partial else ''
    output = dest_dir / f'{safe_name}_audio{suffix}.wav'
    # 音声はマイクコールバックから得た生PCMのため、再サンプリングしない。
    with wave.open(str(output), 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(SAMPLE_RATE)
        wf.writeframes(b''.join(audio_chunks))
    return output


def archive_input_wav(path: Path) -> Path:
    """WAV解析で使用した原本を実行別の音声フォルダにコピーする。"""
    import shutil
    dest_dir = RUN_AUDIO_DIR if RUN_AUDIO_DIR is not None else AUDIO_DIR
    dest_dir.mkdir(parents=True, exist_ok=True)
    output = dest_dir / path.name
    if output.resolve() != path.resolve():
        # 同名ファイルがあれば上書きを避ける。
        if output.exists():
            output = dest_dir / f'{path.stem}_{datetime.now().strftime("%H%M%S_%f")}{path.suffix}'
        shutil.copy2(path, output)
    return output


def save_result(reader: str, gender: str, mode: str, dic: DictionaryData,
                expected: str, source: str, session: RecognitionSession):
    """集計CSVへ追記。実行開始時に区切り行を1つ追加し、実行IDを記録する。"""
    global RUN_DIVIDER_WRITTEN
    RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
    headers = ['日時', 'ファイル名', '読手名', '性別', '実行モード', '使用辞書',
               '前札下の句', '認識札上の句', '設定認識時間(秒)', '最終推定ID',
               '最終類似度', '認識単語', '実認識時間(秒)', '実際の終了予定時刻(秒)', '実行ID']
    if RESULTS_CSV.exists() and RESULTS_CSV.stat().st_size:
        with RESULTS_CSV.open('r', encoding='utf-8-sig', newline='') as f:
            old_reader = csv.reader(f)
            old_header = next(old_reader, [])
            old_rows = list(old_reader)
        if old_header != headers:
            # 古いCSVの列数や並び順に依存せず、列名で対応付けて移行する。
            # 不足する列は空欄（過去の実認識時間などは復元不可）。
            # 旧CSVにしかない追加列も消さずに保持する。
            if not old_header or '日時' not in old_header:
                raise ValueError(f'集計CSVのヘッダーを確認してください: {old_header}')
            extra_headers = [h for h in old_header if h not in headers and h]
            headers += extra_headers
            idx = {h: i for i, h in enumerate(old_header)}
            migrated_rows = [
                [old[idx[h]] if h in idx and idx[h] < len(old) else '' for h in headers]
                for old in old_rows
            ]
            # 既存データのバックアップを残してから安全に置き換える。
            import shutil
            backup = RESULTS_CSV.with_name(RESULTS_CSV.stem + '_before_migration.csv')
            if not backup.exists():
                shutil.copy2(RESULTS_CSV, backup)
            tmp = RESULTS_CSV.with_name(RESULTS_CSV.stem + '_updating.csv')
            with tmp.open('w', encoding='utf-8-sig', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(headers)
                writer.writerows(migrated_rows)
            tmp.replace(RESULTS_CSV)

    new_file = not RESULTS_CSV.exists() or RESULTS_CSV.stat().st_size == 0
    cid, pct = session.result()
    row = [datetime.now().strftime('%Y-%m-%d %H:%M:%S'), source, reader,
           gender, mode, dic.name, '-', expected,
           f'{session.limit:.2f}', cid, f'{pct:.1f}%', session.transcript,
           f'{session.elapsed:.3f}',
           f'{(session.end_at if session.end_at is not None else session.limit):.3f}',
           RUN_ID or '未設定']
    row += [''] * (len(headers) - len(row))
    with RESULTS_CSV.open('a', encoding='utf-8-sig', newline='') as f:
        writer = csv.writer(f)
        if new_file:
            writer.writerow(headers)
        if not RUN_DIVIDER_WRITTEN:
            # CSV自体は罫線やセル色を持てないため、Excelでも一目で分かる区切り行。
            section = [''] * len(headers)
            section[0] = '━━━━━━━━ 今回の実行開始 ━━━━━━━━'
            section[1] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            section[2] = f'辞書: {dic.name}'
            section[3] = f'読手: {reader}'
            section[-1] = RUN_ID or '未設定'
            writer.writerow(section)
            RUN_DIVIDER_WRITTEN = True
        writer.writerow(row)


def save_detail_csv(session: RecognitionSession, source: str, reader: str, gender: str,
                    mode: str, dic: DictionaryData, expected: str = '-') -> Path:
    """1回の認識につき1個の詳細CSV。途中の部分認識・候補順位・時刻を保持。"""
    dest_dir = RUN_DETAIL_DIR if RUN_DETAIL_DIR is not None else DETAIL_DIR
    dest_dir.mkdir(parents=True, exist_ok=True)
    short_stem = make_short_output_stem(source)
    match = re.fullmatch(r'(\d{3})_(\d{8}_\d{6}_\d{6})', short_stem)
    if match:
        number, timestamp = match.groups()
        filename = f'{number}_details{timestamp}.csv'
    else:
        filename = f'{short_stem}_details.csv'
    dest = dest_dir / filename
    base = {'読手名': reader, '性別': gender, '実行モード': mode,
            '使用辞書': dic.name, '入力名': source, '正解札': expected, '実行ID': RUN_ID or '未設定'}
    with dest.open('w', encoding='utf-8-sig', newline='') as f:
        rows = session.detail_rows
        writer = csv.DictWriter(f, fieldnames=list(base) + list(rows[0] if rows else []))
        writer.writeheader()
        for row in rows:
            writer.writerow({**base, **row})
    return dest


def save_rms_plot(session: RecognitionSession, basename: str) -> Path | None:
    """元のWAV解析に準じたRMS画像。マイク・WAV両方で保存する。"""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print('[RMS] matplotlibがないためグラフをスキップしました。')
        return None
    if not session.rms_data:
        return None
    dest_dir = RUN_PLOT_DIR if RUN_PLOT_DIR is not None else PLOT_DIR
    dest_dir.mkdir(parents=True, exist_ok=True)
    times, volumes = zip(*session.rms_data)
    fig, ax = plt.subplots(figsize=(12, 4.5))
    ax.plot(times, volumes, color='#1f77b4', lw=1.2, alpha=.85,
            label='RMS Volume')
    ax.axhline(VOLUME_THRESHOLD, color='red', ls='--', lw=1,
               label=f'Threshold ({VOLUME_THRESHOLD})')
    if session.upper_start is not None:
        ax.axvline(session.upper_start, color='green', ls='-.', lw=1.5,
                   label=f'Uonoku Start ({session.upper_start:.2f}s)')
    if session.upper_start is not None and session.end_at is not None:
        deadline = session.upper_start + session.end_at
        if deadline <= max(times):
            ax.axvline(deadline, color='purple', ls='--', lw=1,
                       label=f'End Threshold ({session.end_at:.2f}s)')
    for marker_t, marker_name in getattr(session, 'phase_markers', []):
        ax.axvline(marker_t, ls='--', lw=1, color='gray', alpha=.8,
                   label=f'{marker_name} ({marker_t:.2f}s)')
    max_rms = max(max(volumes), VOLUME_THRESHOLD * 2, 0.001)
    for i, (t, word) in enumerate(session.word_markers):
        ax.axvline(t, color='darkorange', ls=':', alpha=.55)
        ax.text(t, max_rms * (0.68 if i % 2 else 0.85), f' {word}',
                rotation=90, va='bottom', color='darkorange', fontsize=8)
    cid, _ = session.result()
    ax.set(title=f'RMS Analysis: {basename} (ID: {cid})',
           xlabel='Time (seconds)', ylabel='RMS Volume')
    ax.set_ylim(bottom=0, top=max_rms * 1.35)
    ax.grid(True, ls=':', alpha=.6)
    ax.legend(loc='upper right')
    fig.tight_layout()
    safe_name = make_short_output_stem(basename)
    output = dest_dir / f'{safe_name}_rms.png'
    fig.savefig(output, dpi=120)
    plt.close(fig)
    return output


def process_wav(path: Path, model: Model, dic: DictionaryData, predictor: CardPredictor,
                reader: str, gender: str, limit: float, start_mode: str, mode_label: str, correction: float):
    try:
        with wave.open(str(path), 'rb') as wf:
            if (wf.getnchannels(), wf.getsampwidth(), wf.getframerate(), wf.getcomptype()) != (1, 2, SAMPLE_RATE, 'NONE'):
                print(f'[スキップ] {path.name}: PCM 16bit/16kHz/モノラルWAVが必要です。')
                return
            session = RecognitionSession(create_recognizer(model, dic), predictor, limit, start_mode, False, correction)
            last_upper_voice = None
            upper_end_found = False
            while True:
                data = wf.readframes(CHUNK_SIZE)
                if not data:
                    break
                values = np.frombuffer(data, dtype=np.int16).astype(np.float32)
                dt = len(values) / SAMPLE_RATE
                volume = float(np.sqrt(np.mean(values * values)) / 32768.0) if len(values) else 0.0
                if not session.finished:
                    session.consume(data)
                    if session.upper_start is not None and volume >= VOLUME_THRESHOLD:
                        last_upper_voice = session.total_time
                else:
                    # 判定を止めても、上の句が終了するまでRMS取得を継続。
                    session.total_time += dt
                    session.last_rms = volume
                    session.rms_data.append((session.total_time, volume))
                    if volume >= VOLUME_THRESHOLD:
                        last_upper_voice = session.total_time
                    if (session.elapsed >= UPPER_MIN_TOTAL_SECONDS
                            and last_upper_voice is not None
                            and session.total_time-last_upper_voice >= UPPER_END_SILENCE_SECONDS):
                        session.phase_markers.append((session.total_time, 'Upper Phrase End'))
                        upper_end_found = True
                        break
            if not session.finished:
                session.finish()
            if not upper_end_found:
                print('[注意] WAV内で上の句終了を確認できませんでした。RMS画像は途中までです。')
    except (OSError, EOFError, wave.Error) as exc:
        print(f'[エラー] {path}: {exc}')
        return
    match = re.search(r'set_(\d+)', path.name)
    expected = card_id(match.group(1)) + '_u' if match else '-'
    save_result(reader, gender, mode_label, dic, expected, path.name, session)
    detail_path = save_detail_csv(session, path.name, reader, gender, mode_label, dic, expected)
    print(f'[詳細CSV] {detail_path}')
    print(f'[WAV保存] {archive_input_wav(path)}')
    plot_path = save_rms_plot(session, path.stem)
    if plot_path:
        print(f'[RMS画像] {plot_path}')


def run_microphone(model: Model, dic: DictionaryData, predictor: CardPredictor,
                   reader: str, gender: str, limit: float, start_mode: str,
                   mic_mode: str, device_id: int | None, correction: float):
    """各RMS画像を「前札下の句 -> 間 -> 上の句認識 -> 上の句残り」に統一。

    RMS/録音を保存する時点は「上の句が終わったと判定したとき」。
    次に流れる下の句は次の札の先頭データとして引き継ぐ。
    """
    try:
        import sounddevice as sd
    except ImportError as exc:
        raise RuntimeError('マイク入力には sounddevice が必要です: pip install sounddevice') from exc
    import queue
    audio_queue = queue.Queue(maxsize=120)
    lost_chunks = [0]

    def callback(indata, frames, callback_time, status):
        try:
            audio_queue.put_nowait(bytes(indata))
        except queue.Full:
            lost_chunks[0] += 1

    session = RecognitionSession(create_recognizer(model, dic), predictor, limit,
                                 start_mode, correction=correction, notebook_display=True)
    mode_label = 'マイク単発' if mic_mode == '1' else 'マイク連続'
    count = 0
    audio_chunks = []  # 現在の画像に対応する音声: 前札下の句から上の句終了まで
    recording_name = None
    gate = None  # None=認識処理 / upper_tail / lower_wait / lower_active
    last_speech_at = None
    lower_chunks = []  # 次の札に引き継ぐ「前札下の句」
    lower_rms = []
    lower_elapsed = 0.0
    lower_last_voice = None
    last_wait_render = -1.0
    final_plot_saved = False

    print('Ctrl+C で終了。マイク: 16kHz / モノラル / 16bit')
    try:
        with sd.RawInputStream(samplerate=SAMPLE_RATE, blocksize=CHUNK_SIZE,
                               device=device_id, channels=1, dtype='int16', callback=callback):
            while True:
                try:
                    data = audio_queue.get(timeout=0.5)
                except queue.Empty:
                    continue
                samples = np.frombuffer(data, dtype=np.int16).astype(np.float32)
                dt = len(samples) / SAMPLE_RATE
                rms = float(np.sqrt(np.mean(samples * samples)) / 32768.0) if len(samples) else 0.0
                speaking = rms >= VOLUME_THRESHOLD

                if gate is None:
                    audio_chunks.append(data)
                    session.consume(data)
                    if not session.finished:
                        continue

                    # 認識結果だけを先に確定。画像は「上の句終了」の後に保存する。
                    count += 1
                    recording_name = f'microphone_{datetime.now().strftime("%Y%m%d_%H%M%S_%f")}_{count:03d}'
                    save_result(reader, gender, mode_label, dic, '-', recording_name, session)
                    detail_path = save_detail_csv(session, recording_name, reader, gender, mode_label, dic)
                    print(f'[結果保存] {RESULTS_CSV} / 詳細: {detail_path}')
                    gate = 'upper_tail'
                    session.phase = 'upper_tail'
                    last_speech_at = session.total_time if speaking else session.last_voice_time
                    session.phase_markers.append((session.total_time, 'Recognition End'))
                    session.show_status()
                    continue

                if gate == 'upper_tail':
                    # 上の句の残りが続く間もRMSと音声を記録する。
                    audio_chunks.append(data)
                    session.total_time += dt
                    session.last_rms = rms
                    session.rms_data.append((session.total_time, rms))
                    if speaking:
                        last_speech_at = session.total_time
                    if (session.elapsed >= UPPER_MIN_TOTAL_SECONDS
                            and last_speech_at is not None
                            and session.total_time - last_speech_at >= UPPER_END_SILENCE_SECONDS):
                        session.phase_markers.append((session.total_time, 'Upper Phrase End'))
                        # 下の句がまだ入っていない段階で保存する。
                        wav_path = save_audio_wav(audio_chunks, recording_name)
                        rms_path = save_rms_plot(session, recording_name)
                        print(f'[上の句終了] 録音: {wav_path} / RMS: {rms_path}')
                        audio_chunks = []
                        final_plot_saved = True
                        if mic_mode == '1':
                            break
                        gate = 'lower_wait'
                        session.phase = 'lower_wait'
                        lower_chunks = []
                        lower_rms = []
                        lower_elapsed = 0.0
                        lower_last_voice = None
                        session.show_status()
                        continue

                elif gate == 'lower_wait':
                    session.total_time += dt
                    session.last_rms = rms
                    # 上の句後の間は、今回のRMS画像には含めない。
                    if speaking:
                        gate = 'lower_active'
                        session.phase = 'lower_active'
                        lower_chunks = [data]
                        lower_elapsed = dt
                        lower_rms = [(lower_elapsed, rms)]
                        lower_last_voice = lower_elapsed
                        session.show_status()
                    continue

                elif gate == 'lower_active':
                    session.total_time += dt
                    # この下の句を「次の札の前札下の句」に引き継ぐ。
                    lower_chunks.append(data)
                    lower_elapsed += dt
                    lower_rms.append((lower_elapsed, rms))
                    session.last_rms = rms
                    if speaking:
                        lower_last_voice = lower_elapsed
                    if (lower_last_voice is not None
                            and lower_elapsed - lower_last_voice >= WAIT_SECONDS):
                        # 下の句＋直後の無音を次セットの先頭に持ち越す。
                        session.reset()
                        session.phase = 'upper_wait'
                        session.total_time = lower_elapsed
                        session.rms_data = lower_rms.copy()
                        session.phase_markers = [(0.0, 'Previous Lower Phrase Start'),
                                                 (lower_elapsed, 'Previous Lower Phrase End')]
                        session.last_rms = rms
                        audio_chunks = lower_chunks.copy()
                        lower_chunks = []
                        lower_rms = []
                        lower_elapsed = 0.0
                        lower_last_voice = None
                        recording_name = None
                        gate = None
                        final_plot_saved = False
                        session.show_status('前札下の句終了・次の上の句待機')
                        continue

                if session.total_time - last_wait_render >= 0.2:
                    session.show_status()
                    last_wait_render = session.total_time

    except KeyboardInterrupt:
        print('\n[終了] マイク認識を停止しました。')
    finally:
        # 上の句終了前の手動中断などは、完成画像と混同しないよう _partial にする。
        if gate == 'upper_tail' and recording_name and not final_plot_saved:
            try:
                rms_path = save_rms_plot(session, recording_name + '_partial')
                print(f'[途中終了・不完全なRMS画像] {rms_path}')
            except Exception as exc:
                print(f'[RMS保存失敗] {exc}')
        if audio_chunks:
            try:
                partial_name = recording_name or f'microphone_interrupted_{datetime.now().strftime("%Y%m%d_%H%M%S_%f")}'
                wav_path = save_audio_wav(audio_chunks, partial_name, partial=True)
                print(f'[途中終了時の録音WAV] {wav_path}')
            except Exception as exc:
                print(f'[録音WAV保存失敗] {exc}')
        if lost_chunks[0]:
            print(f'[注意] 音声チャンクの欠落: {lost_chunks[0]}回（処理負荷を確認）')


In [10]:
def main():
    print('=== 競技かるた Vosk 音声認識・札予測 ===')
    model_choice = input('【Model】1: Standard / 2: Small -> ').strip()
    model_path = ROOT / ('model_standard' if model_choice == '1' else 'model_small')
    if not model_path.is_dir():
        print(f'[エラー] Voskモデルが見つかりません: {model_path}')
        return

    reader, gender = reader_prompt()
    print('\n【辞書】')
    for k, (name, path, _) in DICTIONARIES.items():
        print(f' {k}: {name}')
    choice = input('選択 [Enter: 7] -> ').strip() or '7'
    if choice not in DICTIONARIES:
        choice = '7'
    new_mode = '1'
    if choice == '7':
        new_mode = input('【使用語彙】1: 全パーツ / 2: VOSK特化2列のみ [Enter: 1] -> ').strip() or '1'

    source_choice = input('【入力】1: WAVファイル / 2: リアルタイムマイク -> ').strip()
    source_choice = source_choice if source_choice in ('1', '2') else '2'
    custom_id = 'CUSTOM'
    if choice == '6':
        raw = input('【登録札番号】(例: 1 または 001) -> ').strip()
        custom_id = card_id(raw) if raw else 'CUSTOM'
    dictionary = load_dictionary(choice, custom_id, new_mode)
    export_dictionary_csv(dictionary)
    predictor = CardPredictor(dictionary)

    start_mode = input('【認識開始】1: 上の句から直接 / 2: 前札下の句終了後 [Enter: 1] -> ').strip() or '1'
    correction_text = input(f'【認識時間の補正値(秒)】[Enter: {TIME_CORRECTION_SECONDS}] -> ').strip()
    try:
        correction = max(0.0, float(correction_text)) if correction_text else TIME_CORRECTION_SECONDS
    except ValueError:
        correction = TIME_CORRECTION_SECONDS
    limit = input_number('【初語が見つからない場合の安全終了時間(秒)】', DEFAULT_LIMIT_SECONDS)
    print(f'\nVoskモデル読み込み中: {model_path}')
    SetLogLevel(-1)
    model = Model(str(model_path))
    start_run()  # main()の実行ごとに新しい詳細CSVフォルダと実行IDを生成

    if source_choice == '2':
        mode = input('【マイクモード】1: 単発 / 2: 連続 [Enter: 2] -> ').strip() or '2'
        device_text = input('【入力デバイス番号】Enter: OS標準 / l: 一覧表示 -> ').strip().lower()
        if device_text == 'l':
            import sounddevice as sd
            print(sd.query_devices())
            device_text = input('【入力デバイス番号】Enter: OS標準 -> ').strip()
        device_id = int(device_text) if device_text.isdecimal() else None
        run_microphone(model, dictionary, predictor, reader, gender, limit,
                       start_mode, mode, device_id, correction)
        return

    mode = input('【WAVモード】1: 単発 / 2: 連続 / 3: ランダム [Enter: 2] -> ').strip() or '2'
    if mode == '1':
        raw = input('【上の句番号】(例: 1 または 001) -> ').strip()
        cid = card_id(raw)
        files = sorted(RECORD_DIR.glob(f'set_{cid}_*.wav'))
        if not files:
            files = sorted(RECORD_DIR.glob(f'set_{cid}.wav'))
        mode_label = '単発WAVテスト'
    else:
        files = sorted(RECORD_DIR.glob('set_*.wav'))
        if mode == '3':
            random.shuffle(files)
        mode_label = 'ランダムWAVテスト' if mode == '3' else '連続WAVテスト'
    if not files:
        print(f'[エラー] 対象WAVがありません: {RECORD_DIR}')
        return
    print(f'\n対象: {len(files)} 件')
    for i, path in enumerate(files, 1):
        print(f'\n[{i}/{len(files)}] {path.name}')
        process_wav(path, model, dictionary, predictor, reader, gender,
                    limit, start_mode, mode_label, correction)
    print(f'\n[解析完了] 結果: {RESULTS_CSV}')
    print(f'[RMSグラフ] {RUN_PLOT_DIR}')
    print(f'[音声ファイル] {RUN_AUDIO_DIR}')

In [11]:
dictionary = load_dictionary("7", new_mode="1")
export_dictionary_csv(dictionary)

読み込み列: カタカナ決まり字 / ひらがな決まり字 / カタカナ決まり字パーツ / ひらがな決まり字パーツ / カタカナパーツ(1句目) / ひらがなパーツ(1句目) / カタカナパーツ(2句目) / ひらがなパーツ(2句目) / 手動追加パーツ / 誤識別追加 / VOSK特化辞書カタカナ / VOSK特化辞書ひらがな
[辞書ロード完了] 新・かるた統合辞書: 100札 / 1176語 / 2812文字（全札の重複を除外）
[認識時間] CSVの数値は札ごとの最短認識時間です（指定札数: 100）。
[重み付け] 列=True / 単語=True / 札＋単語=True
[辞書内容CSV保存] C:\Users\misak\Desktop\karuta_vosk\results\karuta_dictionary_preview.csv


WindowsPath('c:/Users/misak/Desktop/karuta_vosk/results/karuta_dictionary_preview.csv')

In [14]:
main()

=== 競技かるた Vosk 音声認識・札予測 ===

【辞書】
 1: ひらがな版（統一文字数辞書）
 2: カタカナ版（統一文字数辞書）
 3: 決まり字・手動追加版（サマリー辞書）
 4: VOSK専用辞書
 5: VOSK特化辞書（旧形式）
 6: カスタムテキスト入力（動的辞書）
 7: 新・かるた統合辞書
読み込み列: カタカナ決まり字 / ひらがな決まり字 / カタカナ決まり字パーツ / ひらがな決まり字パーツ / カタカナパーツ(1句目) / ひらがなパーツ(1句目) / カタカナパーツ(2句目) / ひらがなパーツ(2句目) / 手動追加パーツ / 誤識別追加 / VOSK特化辞書カタカナ / VOSK特化辞書ひらがな
[辞書ロード完了] 新・かるた統合辞書: 100札 / 1176語 / 2812文字（全札の重複を除外）
[認識時間] CSVの数値は札ごとの最短認識時間です（指定札数: 100）。
[重み付け] 列=True / 単語=True / 札＋単語=True
[辞書内容CSV保存] C:\Users\misak\Desktop\karuta_vosk\results\karuta_dictionary_preview.csv

Voskモデル読み込み中: c:\Users\misak\Desktop\karuta_vosk\model_small
[今回の実行ID] 20261112_185851_685882
[詳細CSV保存先] c:\Users\misak\Desktop\karuta_vosk\recognition_details\20261112_185851_685882
[RMS画像保存先] c:\Users\misak\Desktop\karuta_vosk\rms_plots\20261112_185851_685882
[録音WAV保存先] c:\Users\misak\Desktop\karuta_vosk\recordings\20261112_185851_685882


HTML(value='<div style="box-sizing:border-box;max-width:780px;padding:5px 9px;\n             border:1px solid …

Ctrl+C で終了。マイク: 16kHz / モノラル / 16bit
[結果保存] c:\Users\misak\Desktop\karuta_vosk\results\batch_recognition_history.csv / 詳細: c:\Users\misak\Desktop\karuta_vosk\recognition_details\20261112_185851_685882\001_details20261112_185910_787527.csv


C:\Users\misak\AppData\Local\Temp\ipykernel_19340\2336210873.py:170: UserWarning: Glyph 31179 (\N{CJK UNIFIED IDEOGRAPH-79CB}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_19340\2336210873.py:170: UserWarning: Glyph 37326 (\N{CJK UNIFIED IDEOGRAPH-91CE}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_19340\2336210873.py:170: UserWarning: Glyph 12398 (\N{HIRAGANA LETTER NO}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_19340\2336210873.py:170: UserWarning: Glyph 12383 (\N{HIRAGANA LETTER TA}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_19340\2336210873.py:170: UserWarning: Glyph 12450 (\N{KATAKANA LETTER A}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_19340\2336210873.py:170: UserWarning: Glyph 12461 (\N{KATAKANA LETTER K

[上の句終了] 録音: c:\Users\misak\Desktop\karuta_vosk\recordings\20261112_185851_685882\001_20261112_185910_787527_audio.wav / RMS: c:\Users\misak\Desktop\karuta_vosk\rms_plots\20261112_185851_685882\001_20261112_185910_787527_rms.png
[結果保存] c:\Users\misak\Desktop\karuta_vosk\results\batch_recognition_history.csv / 詳細: c:\Users\misak\Desktop\karuta_vosk\recognition_details\20261112_185851_685882\002_details20261112_185943_382923.csv


C:\Users\misak\AppData\Local\Temp\ipykernel_19340\2336210873.py:170: UserWarning: Glyph 12399 (\N{HIRAGANA LETTER HA}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_19340\2336210873.py:170: UserWarning: Glyph 12427 (\N{HIRAGANA LETTER RU}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_19340\2336210873.py:170: UserWarning: Glyph 12377 (\N{HIRAGANA LETTER SU}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_19340\2336210873.py:170: UserWarning: Glyph 12366 (\N{HIRAGANA LETTER GI}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_19340\2336210873.py:170: UserWarning: Glyph 12390 (\N{HIRAGANA LETTER TE}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_19340\2336210873.py:173: UserWarning: Glyph 12399 (\N{HIRAGANA LETTER HA}) missing fro

... (133 outputs left out)

C:\Users\misak\AppData\Local\Temp\ipykernel_19340\2336210873.py:170: UserWarning: Glyph 24120 (\N{CJK UNIFIED IDEOGRAPH-5E38}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_19340\2336210873.py:173: UserWarning: Glyph 24120 (\N{CJK UNIFIED IDEOGRAPH-5E38}) missing from font(s) DejaVu Sans.
  fig.savefig(output, dpi=120)


[上の句終了] 録音: c:\Users\misak\Desktop\karuta_vosk\recordings\20261112_185851_685882\093_20261112_193531_825489_audio.wav / RMS: c:\Users\misak\Desktop\karuta_vosk\rms_plots\20261112_185851_685882\093_20261112_193531_825489_rms.png
[結果保存] c:\Users\misak\Desktop\karuta_vosk\results\batch_recognition_history.csv / 詳細: c:\Users\misak\Desktop\karuta_vosk\recognition_details\20261112_185851_685882\094_details20261112_193555_131945.csv
[上の句終了] 録音: c:\Users\misak\Desktop\karuta_vosk\recordings\20261112_185851_685882\094_20261112_193555_131945_audio.wav / RMS: c:\Users\misak\Desktop\karuta_vosk\rms_plots\20261112_185851_685882\094_20261112_193555_131945_rms.png
[結果保存] c:\Users\misak\Desktop\karuta_vosk\results\batch_recognition_history.csv / 詳細: c:\Users\misak\Desktop\karuta_vosk\recognition_details\20261112_185851_685882\095_details20261112_193617_941028.csv
[上の句終了] 録音: c:\Users\misak\Desktop\karuta_vosk\recordings\20261112_185851_685882\095_20261112_193617_941028_audio.wav / RMS: c:\Users\misak\

C:\Users\misak\AppData\Local\Temp\ipykernel_19340\2336210873.py:170: UserWarning: Glyph 12396 (\N{HIRAGANA LETTER NU}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_19340\2336210873.py:170: UserWarning: Glyph 12498 (\N{KATAKANA LETTER HI}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_19340\2336210873.py:173: UserWarning: Glyph 12396 (\N{HIRAGANA LETTER NU}) missing from font(s) DejaVu Sans.
  fig.savefig(output, dpi=120)
C:\Users\misak\AppData\Local\Temp\ipykernel_19340\2336210873.py:173: UserWarning: Glyph 12498 (\N{KATAKANA LETTER HI}) missing from font(s) DejaVu Sans.
  fig.savefig(output, dpi=120)


[上の句終了] 録音: c:\Users\misak\Desktop\karuta_vosk\recordings\20261112_185851_685882\097_20261112_193706_049362_audio.wav / RMS: c:\Users\misak\Desktop\karuta_vosk\rms_plots\20261112_185851_685882\097_20261112_193706_049362_rms.png
[結果保存] c:\Users\misak\Desktop\karuta_vosk\results\batch_recognition_history.csv / 詳細: c:\Users\misak\Desktop\karuta_vosk\recognition_details\20261112_185851_685882\098_details20261112_193730_154883.csv
[上の句終了] 録音: c:\Users\misak\Desktop\karuta_vosk\recordings\20261112_185851_685882\098_20261112_193730_154883_audio.wav / RMS: c:\Users\misak\Desktop\karuta_vosk\rms_plots\20261112_185851_685882\098_20261112_193730_154883_rms.png
[結果保存] c:\Users\misak\Desktop\karuta_vosk\results\batch_recognition_history.csv / 詳細: c:\Users\misak\Desktop\karuta_vosk\recognition_details\20261112_185851_685882\099_details20261112_193755_164190.csv
[上の句終了] 録音: c:\Users\misak\Desktop\karuta_vosk\recordings\20261112_185851_685882\099_20261112_193755_164190_audio.wav / RMS: c:\Users\misak\

修正版

In [17]:
"""競技かるた: VoskによるWAV/マイク音声認識と札予測。

必要: pip install vosk numpy sounddevice matplotlib ipywidgets
モデル: ./model_standard または ./model_small
辞書: ./data/karuta_dictionary.csv (18列の新形式)
"""
from __future__ import annotations

import csv
import math
import glob
import json
import os
import random
import re
import sys
import time
import unicodedata
import wave
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime

try:
    from IPython.display import HTML, display, update_display, clear_output
    from IPython import get_ipython
except ImportError:
    HTML = display = update_display = clear_output = get_ipython = None
from pathlib import Path

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

import numpy as np
from vosk import KaldiRecognizer, Model, SetLogLevel

try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    # Jupyter Notebook / IPython では __file__ が定義されない
    ROOT = Path.cwd()
SAMPLE_RATE = 16000
CHUNK_SIZE = 1600
VOLUME_THRESHOLD = 0.01
WAIT_SECONDS = 0.8  # 下の句の終了判定に必要な連続無音時間
UPPER_END_SILENCE_SECONDS = 1.2  # 上の句の読み終わり判定に必要な連続無音時間
UPPER_MIN_TOTAL_SECONDS = 3.5  # 上の句の開始から、読み終わりと判断しない最短時間
# 読手によって長い間を取る場合は、この2値を調整してください。
DEFAULT_LIMIT_SECONDS = 8.0  # 初語が見つからないときの安全終了時間
TIME_CORRECTION_SECONDS = 0.5  # 最短認識時間への加算値（実行時に変更可）
RESULTS_CSV = ROOT / 'results' / 'batch_recognition_history.csv'
RECORD_DIR = ROOT / 'test_recordings'
PLOT_DIR = ROOT / 'rms_plots'
AUDIO_DIR = ROOT / 'recordings'  # マイク録音とWAV解析元音声の保存先
# 詳細CSVはresultsとは別の場所に、実行ごとに保存。
DETAIL_DIR = ROOT / 'recognition_details'
RUN_ID = None
RUN_DETAIL_DIR = None
RUN_PLOT_DIR = None
RUN_AUDIO_DIR = None
RUN_DIVIDER_WRITTEN = False


def start_run():
    """main()を実行するたびに、実行IDと詳細保存フォルダを作成する。"""
    global RUN_ID, RUN_DETAIL_DIR, RUN_PLOT_DIR, RUN_AUDIO_DIR, RUN_DIVIDER_WRITTEN
    RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    RUN_DETAIL_DIR = DETAIL_DIR / RUN_ID
    RUN_DETAIL_DIR.mkdir(parents=True, exist_ok=True)
    RUN_PLOT_DIR = PLOT_DIR / RUN_ID
    RUN_PLOT_DIR.mkdir(parents=True, exist_ok=True)
    RUN_AUDIO_DIR = AUDIO_DIR / RUN_ID
    RUN_AUDIO_DIR.mkdir(parents=True, exist_ok=True)
    RUN_DIVIDER_WRITTEN = False
    print(f'[今回の実行ID] {RUN_ID}')
    print(f'[詳細CSV保存先] {RUN_DETAIL_DIR}')
    print(f'[RMS画像保存先] {RUN_PLOT_DIR}')
    print(f'[録音WAV保存先] {RUN_AUDIO_DIR}')
    return RUN_ID


DICTIONARIES = {
    '1': ('ひらがな版（統一文字数辞書）', ROOT / 'data/karuta_hiragana_aligned.csv', 'aligned_hiragana'),
    '2': ('カタカナ版（統一文字数辞書）', ROOT / 'data/karuta_katakana_aligned.csv', 'aligned_katakana'),
    '3': ('決まり字・手動追加版（サマリー辞書）', ROOT / 'data/karuta_card_summary_by_kimari_len.csv', 'summary'),
    '4': ('VOSK専用辞書', ROOT / 'data/karuta_vosk_dic.csv', 'vosk_custom'),
    '5': ('VOSK特化辞書（旧形式）', ROOT / 'data/dic_VOSK_cus_converted.csv', 'vosk_cus_converted'),
    '6': ('カスタムテキスト入力（動的辞書）', None, 'custom_input'),
    '7': ('新・かるた統合辞書', ROOT / 'data/karuta_dictionary.csv', 'new_karuta'),
}

# 新辞書の列名。基本は「決まり字 + 分割パーツ + 手動追加 + VOSK特化」。
NEW_COLUMNS = [
    'カタカナ決まり字', 'ひらがな決まり字',
    'カタカナ決まり字パーツ', 'ひらがな決まり字パーツ',
    'カタカナパーツ(1句目)', 'ひらがなパーツ(1句目)',
    'カタカナパーツ(2句目)', 'ひらがなパーツ(2句目)',
    '手動追加パーツ', '誤識別追加',
    'VOSK特化辞書カタカナ', 'VOSK特化辞書ひらがな',
]
# ===== 3種類の重み付けを独立してON/OFFできます =====
# Falseにするとその方式は無効。すべてFalseなら全語1.0倍です。
USE_COLUMN_WEIGHTS = True        # CSVの列ごと
USE_WORD_WEIGHTS = True          # 全札共通・単語ごと
USE_CARD_WORD_WEIGHTS = True    # 特定の札・単語ごと

# ① 列ごとの重み（初期値はすべて1.0）
COLUMN_WEIGHTS = {col: 1.0 for col in NEW_COLUMNS}
COLUMN_WEIGHTS["カタカナ決まり字"] = 1.5
COLUMN_WEIGHTS["ひらがな決まり字"] = 1.5
COLUMN_WEIGHTS["手動追加パーツ"] = 1.0

# ② 単語ごとの重み（札を問わず適用）
WORD_WEIGHTS = {
    '伊佐': 1.5,
    'アキノ': 1.0,
    'カモ': 1.6,
}

# ③ 特定の札＋単語ごとの重み（現在はON）
CARD_WORD_WEIGHTS = {
    ('004', 'ウラ'): 1.6,
    ('023', 'ツキ'): 0.86,
    ('028', 'ヤマ'): 1.3,
    ('032', 'ヤマ'): 1.5,
    ('035', 'ヒト'): 1.5,
    ('056', 'アラ'): 1.6,
    ('063', 'ワタ'): 0.77,
    ('081', 'オト'): 0.8,
    ('099', 'ヒト'): 1.6,
}

# 優先順位：③（札＋単語）> ②（単語）> ①（列）> 1.0
# 3方式をすべてONにした場合も、重みを掛け算しません。
# カタカナ/ひらがな等の表記はnormalize()によって統一して検索します。
# 全ての重みには0以上の有限数を指定してください。

def make_weight_settings():
    """設定の変更を辞書再読込時に反映し、指定語を正規化して検索できるようにする。"""
    def checked_weight(value, where):
        value = float(value)
        if not math.isfinite(value) or value < 0:
            raise ValueError(f'{where}: 重みは0以上の有限数で指定してください: {value}')
        return value

    by_word = {normalize(w): checked_weight(weight, f'WORD_WEIGHTS[{w!r}]')
               for w, weight in WORD_WEIGHTS.items()}
    by_card = {(card_id(cid), normalize(w)): checked_weight(weight, f'CARD_WORD_WEIGHTS[{cid!r}, {w!r}]')
               for (cid, w), weight in CARD_WORD_WEIGHTS.items()}
    by_column = {col: checked_weight(weight, f'COLUMN_WEIGHTS[{col!r}]')
                 for col, weight in COLUMN_WEIGHTS.items()}
    return by_column, by_word, by_card

# ===== 漢字の加点用文字数（ここをON/OFF） =====
USE_HIRAGANA_LENGTH_FOR_KANJI = True  # True: 漢字を読み（ひらがな）に換算した長さで加点

# 自動変換が間違う語はここに正しい読みを記入してください（例：固有名詞）。
KANJI_READING_OVERRIDES = {
    '秋の': 'あきの',
    '秋野': 'あきの',
    '明乃': 'あきの',
    '秋風': 'あきかぜ',
    '月見': 'つきみ',
}

_kakasi_instance = None


def hiragana_reading(word: str) -> str:
    """漢字を含む単語のみ、読みをひらがなにする。変換できなければ元の表記を使う。"""
    global _kakasi_instance
    import unicodedata
    w = str(word)
    if w in KANJI_READING_OVERRIDES:
        return KANJI_READING_OVERRIDES[w]
    if not any('\u3400' <= c <= '\u9fff' or c == '々' for c in w):
        # ひらがな/カタカナは通常の文字数を使う
        return w
    if _kakasi_instance is None:
        try:
            from pykakasi import kakasi
        except ImportError as exc:
            raise ImportError('漢字のひらがな換算をONにするには、Notebookで %pip install pykakasi を実行してください。') from exc
        _kakasi_instance = kakasi()
    try:
        return ''.join(part['hira'] for part in _kakasi_instance.convert(w)) or w
    except Exception:
        return w


def score_length(word: str) -> int:
    """認識語1つの加点基準文字数。漢字換算OFFならこれまでどおり。"""
    if USE_HIRAGANA_LENGTH_FOR_KANJI:
        return len(hiragana_reading(word))
    return len(word)


def normalize(value: object) -> str:
    """Voskの空白・表記揺れによる簡単な取りこぼしを減らす。"""
    return re.sub(r'\s+', '', unicodedata.normalize('NFKC', str(value)).strip()).lower()


def card_id(value: object) -> str:
    s = str(value).strip()
    try:
        return f'{int(s):03d}'
    except ValueError:
        return s


def words_in_cell(value: object) -> list[str]:
    return [normalize(x) for x in str(value or '').replace('　', ' ').split() if normalize(x)]


@dataclass
class DictionaryData:
    name: str
    path: str
    words_by_card: dict[str, dict[str, float]]
    grammar: list[str]
    durations: dict[str, float]  # 札ごとの最短認識時間（最大時間ではない）


def load_dictionary(choice: str, custom_id: str = 'CUSTOM', new_mode: str = '1') -> DictionaryData:
    name, path, kind = DICTIONARIES[choice]
    words_by_card: dict[str, dict[str, float]] = defaultdict(dict)
    durations: dict[str, float] = {}  # 札ごとの最短認識時間
    column_weights, word_weights, card_word_weights = make_weight_settings()

    def add(cid: str, word: str, column: str | None = None):
        word = normalize(word)
        if not word:
            return
        # 基本重みは列設定から取得。同一札内で複数列にあれば最大値。
        base_weight = (column_weights.get(column, 1.0)
                       if USE_COLUMN_WEIGHTS and column else 1.0)
        if USE_WORD_WEIGHTS and word in word_weights:
            base_weight = word_weights[word]
        if USE_CARD_WORD_WEIGHTS and (cid, word) in card_word_weights:
            base_weight = card_word_weights[(cid, word)]
        # 重複除外。列単位の重みが異なる場合は大きいほうを採用。
        # 単語／札＋単語の指定がある場合は同じ値なのでそのまま。
        words_by_card[cid][word] = max(words_by_card[cid].get(word, 0.0), base_weight)

    if kind == 'custom_input':
        raw = input('【単語入力】スペース区切り (例: はるす はる ハルス) -> ')
        for word in words_in_cell(raw):
            add(custom_id, word)
    elif kind == 'new_karuta':
        if not path.exists():
            raise FileNotFoundError(f'辞書がありません: {path}')
        with path.open('r', encoding='utf-8-sig', newline='') as f:
            reader = csv.DictReader(f)
            headers = [h.strip() for h in (reader.fieldnames or [])]
            if '札番号' not in headers:
                raise ValueError('新辞書に「札番号」列がありません。CSVの先頭行を確認してください。')
            allowed = (['VOSK特化辞書カタカナ', 'VOSK特化辞書ひらがな']
                       if new_mode == '2' else NEW_COLUMNS)
            existing = [col for col in allowed if col in headers]
            if not existing:
                raise ValueError('新辞書に利用できる語彙列がありません。列名を確認してください。')
            for raw_row in reader:
                row = {str(k).strip(): v for k, v in raw_row.items() if k is not None}
                cid = card_id(row.get('札番号', ''))
                if not cid or cid in ('000', '00'):
                    continue
                try:
                    durations[cid] = max(0.1, float(row.get('認識時間', '') or DEFAULT_LIMIT_SECONDS))
                except (TypeError, ValueError):
                    pass
                for col in existing:
                    for w in words_in_cell(row.get(col, '')):
                        add(cid, w, column=col)
        print('読み込み列:', ' / '.join(existing))
    else:
        if path is None or not path.exists():
            raise FileNotFoundError(f'辞書がありません: {path}')
        with path.open('r', encoding='utf-8-sig', newline='') as f:
            reader = csv.reader(f)
            next(reader, None)
            for row in reader:
                if not row or row[0].strip() in ('0', '00', ''):
                    continue
                cid = card_id(row[0])
                if kind == 'vosk_cus_converted':
                    cells = row[2:4]
                elif kind == 'vosk_custom':
                    cells = row[3:]
                    if len(row) > 2:
                        try:
                            durations[cid] = max(0.1, float(row[2]))
                        except ValueError:
                            pass
                elif kind in ('aligned_hiragana', 'aligned_katakana'):
                    cells = row[5:6]
                else:
                    cells = row[7:14]
                for cell in cells:
                    for w in words_in_cell(cell):
                        add(cid, w)

    words_by_card = {cid: words for cid, words in words_by_card.items() if words}
    if not words_by_card:
        raise ValueError('辞書に認識できる語彙がありません。')
    grammar = sorted(set(w for words in words_by_card.values() for w in words), key=lambda x: (-len(x), x))
    print(f'[辞書ロード完了] {name}: {len(words_by_card)}札 / {len(grammar)}語 / {sum(map(len, grammar))}文字（全札の重複を除外）')
    print(f'[認識時間] CSVの数値は札ごとの最短認識時間です（指定札数: {len(durations)}）。')
    print(f'[重み付け] 列={USE_COLUMN_WEIGHTS} / 単語={USE_WORD_WEIGHTS} / 札＋単語={USE_CARD_WORD_WEIGHTS}')
    return DictionaryData(name, str(path) if path else '手動入力', words_by_card, grammar, durations)


def export_dictionary_csv(dic: DictionaryData, output: Path | None = None) -> Path:
    """札単位で重複を排除し、辞書の登録内容を1枚1行のCSVに保存する。"""
    output = output or ROOT / 'results' / 'karuta_dictionary_preview.csv'
    output.parent.mkdir(parents=True, exist_ok=True)
    f = output.open('w', encoding='utf-8-sig', newline='')
    with f:
        writer = csv.writer(f)
        writer.writerow(['札番号', '最短認識時間(秒)', '重複なし単語数',
                         '重複なし総文字数', '加点用総文字数', '平均単語長', '最短単語長', '最長単語長', '登録単語一覧', '単語別加点用文字数', '単語別重み', '加重満点スコア'])
        for cid in sorted(dic.words_by_card):
            words = sorted(dic.words_by_card[cid], key=lambda w: (-len(w), w))
            lengths = [len(w) for w in words]
            writer.writerow([cid, dic.durations.get(cid, ''), len(words), sum(lengths),
                             sum(score_length(w) for w in words),
                             round(sum(lengths) / len(words), 2) if words else 0,
                             min(lengths, default=0), max(lengths, default=0), ' '.join(words),
                             ' '.join(f'{w}:{score_length(w)}' for w in words),
                             ' '.join(f'{w}:{dic.words_by_card[cid][w]:g}' for w in words),
                             round(sum(score_length(w) * dic.words_by_card[cid][w] for w in words), 4)])
    print(f'[辞書内容CSV保存] {output.resolve()}')
    return output

In [2]:
class CardPredictor:
    """部分認識を毎回再計算するため、同じ語で点数が二重加算されない。"""
    def __init__(self, dictionary: DictionaryData):
        self.dictionary = dictionary
        self.index: dict[str, list[tuple[str, float]]] = defaultdict(list)
        self.max_scores = {}
        for cid, words in dictionary.words_by_card.items():
            self.max_scores[cid] = sum(score_length(w) * weight for w, weight in words.items()) or 1.0
            for word, weight in words.items():
                self.index[word].append((cid, weight))
        self.all_words = sorted(self.index, key=lambda x: (-len(x), x))

    def predict(self, transcript: str) -> tuple[list[tuple[str, float, float]], list[str]]:
        compact = normalize(transcript)
        if not compact:
            return [], []
        # まず語単位の一致。それ以外に、Voskが単語を連結した出力も拾う。
        tokens = set(words_in_cell(transcript))
        matched = set(w for w in tokens if w in self.index)
        covered = [False] * len(compact)
        for w in sorted(matched, key=len, reverse=True):
            at = compact.find(w)
            while at >= 0:
                for i in range(at, at + len(w)):
                    covered[i] = True
                at = compact.find(w, at + 1)
        for w in self.all_words:
            if w in matched or len(w) < 2:
                continue
            at = compact.find(w)
            while at >= 0:
                if not any(covered[at:at + len(w)]):
                    matched.add(w)
                    for i in range(at, at + len(w)):
                        covered[i] = True
                    break
                at = compact.find(w, at + 1)

        scores = defaultdict(float)
        for w in matched:
            for cid, weight in self.index[w]:
                scores[cid] += score_length(w) * weight
        ranking = sorted(((cid, min(100.0, 100.0 * score / self.max_scores[cid]), score)
                          for cid, score in scores.items()), key=lambda x: (-x[1], -x[2], x[0]))
        return ranking, sorted(matched, key=lambda x: (-len(x), x))


def reader_prompt() -> tuple[str, str]:
    choice = input('【読手】1: 稲葉専任 / 2: 芹野専任 / 3: その他 -> ').strip()
    if choice == '1':
        return '稲葉専任読手', '男性'
    if choice == '2':
        return '芹野専任読手', '女性'
    name = input(' -> 読手名を入力: ').strip() or '未指定読手'
    gender_choice = input(' -> 性別 1:男性 / 2:女性 / 3:未指定 -> ').strip()
    return name, {'1': '男性', '2': '女性'}.get(gender_choice, '未指定')


def input_number(prompt: str, default: float) -> float:
    raw = input(f'{prompt} [Enter: {default}] -> ').strip()
    if not raw:
        return default
    try:
        return max(0.1, float(raw))
    except ValueError:
        print('数値ではないため規定値を使用します。')
        return default


def create_recognizer(model: Model, dictionary: DictionaryData) -> KaldiRecognizer:
    grammar = dictionary.grammar + ['[unk]']
    rec = KaldiRecognizer(model, SAMPLE_RATE, json.dumps(grammar, ensure_ascii=False))
    return rec


def safe_vosk_text(result: str, key: str = 'text') -> str:
    try:
        return str(json.loads(result).get(key, '') or '')
    except (ValueError, TypeError):
        return ''

In [3]:
class NotebookStatus:
    """Notebookで高さを抑えた固定ダッシュボードを表示・上書きする。"""
    def __init__(self, enabled: bool = True):
        self.enabled = bool(enabled and display is not None)
        self.widget = widgets.HTML(value='') if self.enabled and widgets is not None else None
        self.created = False
        self.display_id = 'karuta_dashboard_' + datetime.now().strftime('%H%M%S_%f')

    def show(self, session, message: str = ''):
        if not self.enabled:
            return
        from html import escape
        def h(value):
            return escape(str(value))

        elapsed = session.elapsed
        target = session.end_at if session.end_at is not None else session.limit
        remaining = max(0.0, target - elapsed)
        phase = {'lower': ('前札の下の句中' if session.voice_seen else '前札の下の句待ち'),
                 'upper_wait': '次の上の句待ち',
                 'upper_active': '認識中',
                 'upper_tail': '上の句の残りを待機中',
                 'lower_wait': '下の句待ち（間奏）',
                 'lower_active': '前札の下の句中',
                 'lower_silence': '下の句終了確認中'}.get(session.phase, session.phase)
        if session.finished and session.phase == 'upper_active':
            phase = '判定完了'
        if message:
            phase += '・' + message

        # 上位5札を固定5行で表示する。棒はCSSで描き、高さを抑える。
        entries = []
        for rank in range(5):
            if rank < len(session.ranking):
                cid, sim, pts = session.ranking[rank]
                maximum = session.predictor.max_scores.get(cid, 1.0)
                pct = max(0, min(100, sim))
                entries.append(
                    f'<tr><td>{rank+1}</td><td><b>{h(cid)}番</b></td>'
                    f'<td>{pts:g}/{maximum:g}</td><td>{sim:.1f}%</td>'
                    f'<td><div style="height:7px;background:#d9e0e6;border-radius:4px;overflow:hidden">'
                    f'<div style="height:100%;width:{pct:.1f}%;background:#16886f"></div></div></td></tr>'
                )
            else:
                entries.append(
                    f'<tr><td>{rank+1}</td><td>---番</td><td>0/---</td><td>0.0%</td>'
                    f'<td><div style="height:7px;background:#d9e0e6;border-radius:4px"></div></td></tr>'
                )

        # 画面には最新3件を1行表示する。全履歴は詳細CSVに保存する。
        logs = getattr(session, 'recognition_logs', [])[-3:]
        log_text = '　｜　'.join(f'{t:.2f}s {h(w)}→{h(card)}番' for t, w, card, _ in logs)
        if not log_text:
            log_text = 'まだ一致語なし'
        transcript = h(session.transcript or '（未認識）')
        first = h(session.first_word or '---')
        ref_card = h(session.first_card or '---')
        timing_state = h('確定' if session.timing_fixed else ('仮設定' if session.first_word else '未設定'))
        timing_candidates = len(session.timing_candidates)
        html = f"""<div style="box-sizing:border-box;max-width:780px;padding:5px 9px;
             border:1px solid #c4cbd3;border-radius:5px;background:#fff;
             color:#1d2939;font:12px/1.25 sans-serif">
          <div style="display:flex;justify-content:space-between;gap:8px;align-items:center;
                      margin:0 0 3px 0;white-space:nowrap">
            <b style="font-size:13px">競技かるた 札予測</b>
            <span>状態：<b>{h(phase)}</b></span>
          </div>
          <div style="padding:3px 5px;background:#f0f3f6;white-space:nowrap;overflow:hidden">
            経過 <b>{elapsed:.2f}s</b>　終了 <b>{target:.2f}s</b>　残り <b>{remaining:.2f}s</b>
            　RMS <b>{session.last_rms:.4f}</b>
          </div>
          <div style="padding:3px 0;white-space:nowrap;overflow:hidden;text-overflow:ellipsis">
            <b>認識：</b>{transcript}
          </div>
          <div style="padding:0 0 3px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;color:#526071">
            初語 {first}　時間基準 {ref_card}番（{timing_state}・候補{timing_candidates}枚）　補正 +{session.correction:.2f}s
          </div>
          <table style="table-layout:fixed;width:100%;border-collapse:collapse;
                        font:12px/1.15 sans-serif;text-align:left">
            <thead><tr style="background:#e9eef3">
              <th style="width:8%">順位</th><th style="width:17%">札</th>
              <th style="width:20%">点/満点</th><th style="width:16%">一致率</th>
              <th style="width:39%">スコア</th>
            </tr></thead>
            <tbody>{''.join(entries)}</tbody>
          </table>
          <div style="margin-top:3px;padding-top:3px;border-top:1px solid #d6dce3;
                      white-space:nowrap;overflow:hidden;text-overflow:ellipsis">
            <b>直近：</b>{log_text}
          </div>
        </div>"""
        # ウィジェット1つだけを更新し、過去の表示を積み重ねない。
        if self.widget is not None:
            self.widget.value = html
            if not self.created:
                display(self.widget)
                self.created = True
        elif not self.created:
            display(HTML(html), display_id=self.display_id)
            self.created = True
        else:
            update_display(HTML(html), display_id=self.display_id)


class RecognitionSession:
    """マイク音声を処理し、Notebook表示と途中経過ログを蓄積する。"""
    def __init__(self, rec: KaldiRecognizer, predictor: CardPredictor, limit: float,
                 start_mode: str, show_partial: bool = True,
                 correction: float = TIME_CORRECTION_SECONDS,
                 notebook_display: bool = False):
        self.rec = rec
        self.predictor = predictor
        self.limit = limit
        self.correction = correction
        self.start_mode = start_mode
        self.show_partial = show_partial
        self.dashboard = NotebookStatus(enabled=notebook_display)
        self.notebook_display = bool(self.dashboard.enabled)
        self.reset()

    @property
    def elapsed(self):
        return (self.total_time - self.upper_start) if self.upper_start is not None else 0.0

    def reset(self):
        self.rec.Reset()
        self.phase = 'lower' if self.start_mode == '2' else 'upper_wait'
        self.total_time = 0.0
        self.last_voice_time = None
        self.voice_seen = False
        self.committed = []
        self.transcript = ''
        self.ranking = []
        self.matched = []
        self.last_display = -1.0
        self.rms_data = []
        self.upper_start = None
        self.finished = False
        self.first_word = None
        self.first_card = None
        self.first_word_at = None
        self.timing_candidates = set()  # 初語に一致する札の候補集合
        self.timing_seen_words = set()  # 時間再設定に使用済みの語
        self.timing_fixed = False  # 1枚まで絞れたときのみ確定
        self.timing_revisions = 0
        self.end_at = None
        self.detail_rows = []
        self.word_markers = []  # (音声先頭からの秒数, 単語)
        self.phase_markers = []  # (音声先頭からの秒数, 状態名)
        self.recognition_logs = []  # (上の句経過秒数, 語, 札番号, 加点)
        self.last_rms = 0.0
        if self.notebook_display:
            self.dashboard.show(self)
        else:
            print('\n[待機] ' + ('前札の下の句を読んでください。無音後に上の句を認識します。'
                                   if self.phase == 'lower' else '上の句を読んでください。'))

    def record_event(self, event: str):
        top = self.ranking[:3]
        row = {
            'イベント': event,
            '全体経過時間(秒)': round(self.total_time, 3),
            '上の句経過時間(秒)': round(self.elapsed, 3),
            '音量RMS': round(self.last_rms, 6),
            '状態': self.phase,
            '認識文字列': self.transcript,
            '一致語': ' '.join(self.matched),
            '初認識語': self.first_word or '',
            '初認識時刻(秒)': round(self.first_word_at, 3) if self.first_word_at is not None else '',
            '基準札番号': self.first_card or '',
            '基準札最短時間(秒)': self.predictor.dictionary.durations.get(self.first_card, '') if self.first_card else '',
            '時間候補札': ' '.join(sorted(self.timing_candidates)),
            '時間基準確定': self.timing_fixed,
            '時間再設定回数': self.timing_revisions,
            '補正値(秒)': self.correction,
            '終了予定(秒)': self.end_at if self.end_at is not None else self.limit,
            '終了まで(秒)': round(max(0.0, (self.end_at if self.end_at is not None else self.limit) - self.elapsed), 3),
        }
        for i in range(3):
            cid, pct, score = top[i] if i < len(top) else ('', '', '')
            row.update({f'第{i+1}候補札': cid, f'第{i+1}類似度(%)': round(pct, 2) if pct != '' else '',
                        f'第{i+1}スコア': round(score, 3) if score != '' else ''})
        # イベント発生時の全候補を保存（後で順位を再計算しない）。
        # 各札の加点と満点を保存するので、CSVで当時の順位推移を検証できる。
        row['_候補全件'] = [
            {'順位': rank, '札番号': cid, '加点': round(score, 4),
             '満点': round(self.predictor.max_scores.get(cid, 0.0), 4),
             '類似度(%)': round(sim, 3)}
            for rank, (cid, sim, score) in enumerate(self.ranking, 1)
        ]
        self.detail_rows.append(row)

    def show_status(self, message=''):
        if self.notebook_display:
            self.dashboard.show(self, message)
        elif self.show_partial and self.phase == 'upper_active':
            labels = ' / '.join(f'{cid}番 {pct:.1f}%' for cid, pct, _ in self.ranking[:3]) or '候補なし'
            print(f'\r[予測] {labels} | 認識: {self.transcript[:65]}', end=' '*10 + '\r', flush=True)

    def _set_timing_from_candidates(self):
        """候補札の最短時間の最大値を仮採用。1札になれば終了時刻を確定する。"""
        if not self.timing_candidates:
            return
        durations = self.predictor.dictionary.durations
        self.first_card = sorted(self.timing_candidates,
                                 key=lambda cid: (-durations.get(cid, self.limit), cid))[0]
        min_seconds = durations.get(self.first_card, self.limit)
        self.end_at = max(0.0, min_seconds + self.correction)
        self.timing_fixed = len(self.timing_candidates) == 1

    def _update(self, partial: str, force: bool = False, event: str = '部分認識'):
        all_text = ' '.join(self.committed + ([partial] if partial else []))
        if all_text == self.transcript and not force:
            return
        self.transcript = all_text
        self.ranking, self.matched = self.predictor.predict(all_text)
        # 認識された新しい辞書語を、RMSグラフに表示する時刻付きで記録する。
        already = {word for _, word in self.word_markers}
        for word in self.matched:
            if word not in already:
                self.word_markers.append((self.total_time, word))
                if self.ranking:
                    for cid, _, _ in self.ranking:
                        if any(k == cid for k, _ in self.predictor.index.get(word, [])):
                            weight = dict(self.predictor.index[word])[cid]
                            self.recognition_logs.append((self.elapsed, word, cid, score_length(word)*weight))
                            break
        # 時間基準を決めるための語を、音声文字列に現れた順に処理する。
        # 同じ語が部分認識で繰り返されても、再設定・加点は重複させない。
        word_stream = [w for w in words_in_cell(all_text)
                       if w != '[unk]' and w in self.predictor.index]
        compact = normalize(all_text.replace('[unk]', ''))
        if not word_stream and self.matched:
            # 空白なしで連結認識された場合は、重なりのないパーツのみを採用。
            # 「あきの」とその部分文字列「あき」を別々の2語と数えない。
            occupied = set()
            ordered = sorted(self.matched, key=lambda w: (compact.find(w), -len(w), w))
            for w in ordered:
                start = compact.find(w)
                if start < 0 or any(i in occupied for i in range(start, start + len(w))):
                    continue
                word_stream.append(w)
                occupied.update(range(start, start + len(w)))

        for token in word_stream:
            if token in self.timing_seen_words:
                continue
            self.timing_seen_words.add(token)
            word_candidates = {cid for cid, _ in self.predictor.index[token]}
            if self.first_word is None:
                self.first_word = token
                self.first_word_at = self.elapsed
                self.timing_candidates = word_candidates
                self._set_timing_from_candidates()
                self.record_event('初語・時間仮設定' if not self.timing_fixed else '初語・時間確定')
                if not self.notebook_display:
                    print(f'\n[初語] {token}: {len(word_candidates)}札 / '
                          f'基準{self.first_card}番 / 終了予定{self.end_at:.2f}秒')
            elif not self.timing_fixed:
                narrowed = self.timing_candidates & word_candidates
                if narrowed and narrowed != self.timing_candidates:
                    self.timing_candidates = narrowed
                    self.timing_revisions += 1
                    self._set_timing_from_candidates()
                    self.record_event('追加語・時間確定' if self.timing_fixed else '追加語・時間再設定')
                    if not self.notebook_display:
                        print(f'\n[時間再設定] {token}: 候補{len(narrowed)}札 / '
                              f'基準{self.first_card}番 / 終了予定{self.end_at:.2f}秒')
                # 共通候補がない追加語で時間を変更しない（誤認識による揺れを防ぐ）。
        self.record_event(event)
        if self.show_partial and (force or self.total_time - self.last_display >= 0.20):
            self.show_status()
            self.last_display = self.total_time

    def consume(self, data: bytes):
        if self.finished or not data:
            return
        values = np.frombuffer(data, dtype=np.int16).astype(np.float32)
        volume = float(np.sqrt(np.mean(values * values)) / 32768.0) if len(values) else 0.0
        dt = len(values) / SAMPLE_RATE
        self.total_time += dt
        self.last_rms = volume
        self.rms_data.append((self.total_time, volume))
        speaking = volume >= VOLUME_THRESHOLD

        if self.phase == 'lower':
            if speaking:
                self.voice_seen = True
                self.last_voice_time = self.total_time
            elif self.voice_seen and self.last_voice_time is not None and self.total_time - self.last_voice_time >= WAIT_SECONDS:
                self.phase = 'upper_wait'
                self.voice_seen = False
                self.last_voice_time = None
                self.rec.Reset()
                self.record_event('下の句終了')
                if self.notebook_display:
                    self.show_status('上の句待機')
                else:
                    print('\n[切替] 下の句が終了。次の上の句を待機します。')
            return

        if self.phase == 'upper_wait':
            if not speaking:
                return
            self.phase = 'upper_active'
            self.upper_start = self.total_time - dt
            self.voice_seen = True
            self.record_event('認識開始')
            if self.notebook_display:
                self.show_status()
            else:
                print('\n[認識開始]')

        if speaking:
            self.last_voice_time = self.total_time
        if self.rec.AcceptWaveform(data):
            text = safe_vosk_text(self.rec.Result())
            if text:
                self.committed.append(text)
            self._update('', force=True, event='確定認識')
        else:
            self._update(safe_vosk_text(self.rec.PartialResult(), 'partial'))
        # 文字起こしが更新されなくても、経過時間だけは約0.2秒ごとに表示更新する。
        if self.notebook_display and self.total_time - self.last_display >= 0.20:
            self.show_status()
            self.last_display = self.total_time

        if self.end_at is not None:
            if self.elapsed >= self.end_at:
                self.finish()
        elif self.elapsed >= self.limit:
            self.finish()

    def finish(self):
        if self.finished:
            return
        text = safe_vosk_text(self.rec.FinalResult())
        if text:
            self.committed.append(text)
        self._update('', force=True, event='最終文字起こし')
        self.finished = True
        self.record_event('認識終了')
        if self.notebook_display:
            self.show_status('結果確定')
        else:
            print('\n' + '-' * 68)
            if self.ranking:
                print('最終候補: ' + ' / '.join(f'{cid}番 ({pct:.1f}%)' for cid, pct, _ in self.ranking[:5]))
            else:
                print('最終候補: なし（音声・辞書・マイク音量を確認）')
            print(f'認識文字列: {self.transcript or "なし"}')
            print(f'時間基準: 初語={self.first_word or "なし"} / 基準札={self.first_card or "なし"} / 終了時刻={self.end_at if self.end_at is not None else self.limit:.2f}秒 / {"確定" if self.timing_fixed else "仮設定"}')
            print('一致語: ' + (' '.join(self.matched) if self.matched else 'なし'))
            print('-' * 68)

    def result(self):
        winner = self.ranking[0] if self.ranking else ('---', 0.0, 0.0)
        return winner[0], winner[1]

In [4]:
def make_short_output_stem(source: str) -> str:
    """マイク名から「連番_日時」に変換する。WAV入力では元名を利用する。"""
    stem = Path(source).stem
    match = re.fullmatch(r'microphone_(\d{8}_\d{6}_\d{6})_(\d{3})', stem)
    if match:
        stamp, number = match.groups()
        return f'{number}_{stamp}'
    return re.sub(r'[^\w.-]', '_', stem)[:80]


def save_audio_wav(audio_chunks: list[bytes], basename: str, *, partial: bool = False) -> Path | None:
    """マイクから取得したPCM 16-bit/16kHz/モノラル音声をWAVとして保存する。"""
    if not audio_chunks:
        return None
    dest_dir = RUN_AUDIO_DIR if RUN_AUDIO_DIR is not None else AUDIO_DIR
    dest_dir.mkdir(parents=True, exist_ok=True)
    safe_name = make_short_output_stem(basename)
    suffix = '_partial' if partial else ''
    output = dest_dir / f'{safe_name}_audio{suffix}.wav'
    # 音声はマイクコールバックから得た生PCMのため、再サンプリングしない。
    with wave.open(str(output), 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(SAMPLE_RATE)
        wf.writeframes(b''.join(audio_chunks))
    return output


def archive_input_wav(path: Path) -> Path:
    """WAV解析で使用した原本を実行別の音声フォルダにコピーする。"""
    import shutil
    dest_dir = RUN_AUDIO_DIR if RUN_AUDIO_DIR is not None else AUDIO_DIR
    dest_dir.mkdir(parents=True, exist_ok=True)
    output = dest_dir / path.name
    if output.resolve() != path.resolve():
        # 同名ファイルがあれば上書きを避ける。
        if output.exists():
            output = dest_dir / f'{path.stem}_{datetime.now().strftime("%H%M%S_%f")}{path.suffix}'
        shutil.copy2(path, output)
    return output


def save_result(reader: str, gender: str, mode: str, dic: DictionaryData,
                expected: str, source: str, session: RecognitionSession):
    """集計CSVへ追記。実行開始時に区切り行を1つ追加し、実行IDを記録する。"""
    global RUN_DIVIDER_WRITTEN
    RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
    headers = ['日時', 'ファイル名', '読手名', '性別', '実行モード', '使用辞書',
               '前札下の句', '認識札上の句', '設定認識時間(秒)', '最終推定ID',
               '最終類似度', '認識単語', '実認識時間(秒)', '実際の終了予定時刻(秒)', '実行ID']
    if RESULTS_CSV.exists() and RESULTS_CSV.stat().st_size:
        with RESULTS_CSV.open('r', encoding='utf-8-sig', newline='') as f:
            old_reader = csv.reader(f)
            old_header = next(old_reader, [])
            old_rows = list(old_reader)
        if old_header != headers:
            # 古いCSVの列数や並び順に依存せず、列名で対応付けて移行する。
            # 不足する列は空欄（過去の実認識時間などは復元不可）。
            # 旧CSVにしかない追加列も消さずに保持する。
            if not old_header or '日時' not in old_header:
                raise ValueError(f'集計CSVのヘッダーを確認してください: {old_header}')
            extra_headers = [h for h in old_header if h not in headers and h]
            headers += extra_headers
            idx = {h: i for i, h in enumerate(old_header)}
            migrated_rows = [
                [old[idx[h]] if h in idx and idx[h] < len(old) else '' for h in headers]
                for old in old_rows
            ]
            # 既存データのバックアップを残してから安全に置き換える。
            import shutil
            backup = RESULTS_CSV.with_name(RESULTS_CSV.stem + '_before_migration.csv')
            if not backup.exists():
                shutil.copy2(RESULTS_CSV, backup)
            tmp = RESULTS_CSV.with_name(RESULTS_CSV.stem + '_updating.csv')
            with tmp.open('w', encoding='utf-8-sig', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(headers)
                writer.writerows(migrated_rows)
            tmp.replace(RESULTS_CSV)

    new_file = not RESULTS_CSV.exists() or RESULTS_CSV.stat().st_size == 0
    cid, pct = session.result()
    row = [datetime.now().strftime('%Y-%m-%d %H:%M:%S'), source, reader,
           gender, mode, dic.name, '-', expected,
           f'{session.limit:.2f}', cid, f'{pct:.1f}%', session.transcript,
           f'{session.elapsed:.3f}',
           f'{(session.end_at if session.end_at is not None else session.limit):.3f}',
           RUN_ID or '未設定']
    row += [''] * (len(headers) - len(row))
    with RESULTS_CSV.open('a', encoding='utf-8-sig', newline='') as f:
        writer = csv.writer(f)
        if new_file:
            writer.writerow(headers)
        if not RUN_DIVIDER_WRITTEN:
            # CSV自体は罫線やセル色を持てないため、Excelでも一目で分かる区切り行。
            section = [''] * len(headers)
            section[0] = '━━━━━━━━ 今回の実行開始 ━━━━━━━━'
            section[1] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            section[2] = f'辞書: {dic.name}'
            section[3] = f'読手: {reader}'
            section[-1] = RUN_ID or '未設定'
            writer.writerow(section)
            RUN_DIVIDER_WRITTEN = True
        writer.writerow(row)


def save_detail_csv(session: RecognitionSession, source: str, reader: str, gender: str,
                    mode: str, dic: DictionaryData, expected: str = '-') -> Path:
    """順位ごと1行の詳細CSVと、色付きHTMLを同じ実行日時フォルダに保存。"""
    from html import escape
    dest_dir = RUN_DETAIL_DIR if RUN_DETAIL_DIR is not None else DETAIL_DIR
    dest_dir.mkdir(parents=True, exist_ok=True)
    short_stem = make_short_output_stem(source)
    match = re.fullmatch(r'(\d{3})_(\d{8}_\d{6}_\d{6})', short_stem)
    if match:
        number, timestamp = match.groups()
        filename = f'{number}_details{timestamp}.csv'
    else:
        filename = f'{short_stem}_details.csv'
    dest = dest_dir / filename

    fields = [
        '実行ID', '入力名', '読手名', '性別', '実行モード', '使用辞書', '正解札',
        'イベント番号', 'イベント', '状態', '全体経過時間(秒)', '上の句経過時間(秒)',
        '音量RMS', '認識文字列', '一致語', '初認識語', '初認識時刻(秒)',
        '時間候補札', '時間基準確定', '時間再設定回数', '基準札番号',
        '基準札最短時間(秒)', '補正値(秒)', '終了予定(秒)', '終了まで(秒)',
        '順位', '札番号', '加点', '満点', '類似度(%)'
    ]
    info = {'実行ID': RUN_ID or '未設定', '入力名': source, '読手名': reader,
            '性別': gender, '実行モード': mode, '使用辞書': dic.name, '正解札': expected}
    rows = []
    for event_no, snapshot in enumerate(session.detail_rows, start=1):
        candidates = snapshot.get('_候補全件')
        if candidates is None:
            # 古いメモリ状態で呼ばれた場合も保存できるようにする。
            candidates = []
            for rank in range(1, 4):
                card = snapshot.get(f'第{rank}候補札', '')
                if card:
                    maximum = session.predictor.max_scores.get(card, '')
                    candidates.append({'順位': rank, '札番号': card,
                                       '加点': snapshot.get(f'第{rank}スコア', ''),
                                       '満点': round(maximum, 4) if maximum != '' else '',
                                       '類似度(%)': snapshot.get(f'第{rank}類似度(%)', '')})
        # 候補がないイベントも消さずに1行残す。
        for candidate in (candidates or [{}]):
            rows.append({**info, 'イベント番号': event_no,
                         **{key: snapshot.get(key, '') for key in fields if key in snapshot},
                         **candidate})
    with dest.open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

    # CSVは色・罫線を保持できないため、同一データの読みやすいHTMLレポートも作成。
    report = dest.with_suffix('.html')
    def fmt(value):
        return escape(str(value if value is not None else ''))
    final_id, final_pct = session.result()
    blocks = []
    for event_no, snapshot in enumerate(session.detail_rows, start=1):
        candidates = snapshot.get('_候補全件')
        if candidates is None:
            candidates = [dict(順位=i, 札番号=snapshot.get(f'第{i}候補札', ''),
                               加点=snapshot.get(f'第{i}スコア', ''),
                               満点=session.predictor.max_scores.get(snapshot.get(f'第{i}候補札', ''), ''),
                               **{'類似度(%)': snapshot.get(f'第{i}類似度(%)', '')})
                          for i in range(1, 4) if snapshot.get(f'第{i}候補札')]
        table_rows = []
        for cand in candidates[:5]:
            pct = cand.get('類似度(%)', 0) or 0
            pct_float = max(0.0, min(100.0, float(pct)))
            cls = 'first' if cand.get('順位') == 1 else ''
            table_rows.append(f"<tr class='{cls}'><td>{fmt(cand.get('順位', ''))}</td>"
                f"<td>{fmt(cand.get('札番号', ''))}</td><td>{fmt(cand.get('加点', ''))}</td>"
                f"<td>{fmt(cand.get('満点', ''))}</td><td><b>{pct_float:.1f}%</b></td>"
                f"<td><div class='bar'><div style='width:{pct_float:.1f}%'></div></div></td></tr>")
        if not table_rows:
            table_rows = ["<tr><td colspan='6' class='empty'>一致する候補札なし</td></tr>"]
        blocks.append(f"<section><div class='event'>"
            f"<span class='seq'>#{event_no:03}</span><b>{fmt(snapshot.get('イベント', ''))}</b>"
            f"<span class='muted'>上の句 {fmt(snapshot.get('上の句経過時間(秒)', ''))} 秒"
            f" ／ RMS {fmt(snapshot.get('音量RMS', ''))} ／ 状態 {fmt(snapshot.get('状態', ''))}</span></div>"
            f"<div class='speech'>認識：{fmt(snapshot.get('認識文字列', '')) or '（未認識）'}</div>"
            f"<div class='muted small'>一致語：{fmt(snapshot.get('一致語', '')) or 'なし'} ／ "
            f"基準札：{fmt(snapshot.get('基準札番号', '')) or '未設定'} ／ "
            f"終了予定：{fmt(snapshot.get('終了予定(秒)', ''))} 秒</div>"
            f"<table><thead><tr><th>順位</th><th>札番号</th><th>加点</th><th>満点</th>"
            f"<th>類似度</th><th>比率</th></tr></thead><tbody>{''.join(table_rows)}</tbody></table></section>")
    css = """body{font-family:"Yu Gothic","Meiryo",sans-serif;background:#f4f7fb;color:#243349;margin:0;padding:20px}
    main{max-width:970px;margin:auto}header{background:#193b57;color:#fff;padding:18px 22px;border-radius:10px}
    header h1{font-size:20px;margin:0 0 8px}.meta{font-size:12px;line-height:1.8}
    .summary{background:#fff;border:1px solid #dce5ee;border-radius:8px;padding:12px 16px;margin:14px 0;font-weight:bold}
    section{background:#fff;border:1px solid #d4e0ed;border-radius:8px;overflow:hidden;margin:12px 0}
    .event{background:#e9f1f8;padding:8px 12px;display:flex;gap:12px;flex-wrap:wrap;align-items:center}
    .seq{color:#0e5d95;font-weight:bold}.muted{color:#61758d;font-weight:normal}.small{font-size:12px;padding:0 12px 8px}
    .speech{padding:9px 12px;font-weight:bold}table{border-collapse:collapse;width:100%;font-size:13px}
    th,td{border:1px solid #dce5ec;padding:5px 9px;text-align:center}th{background:#edf3f8;color:#314d68}
    tr:nth-child(even){background:#f8fafc}tr.first{background:#e3f4eb;font-weight:bold}
    .bar{width:100%;height:9px;background:#e1e9ee;border-radius:5px;overflow:hidden}.bar div{height:100%;background:#159772}
    .empty{color:#8592a1}.note{font-size:12px;color:#61758d;margin:12px 0}
    @media print{body{padding:0;background:white}section{break-inside:avoid}}
    """
    html_text = ('<!doctype html><html lang="ja"><head><meta charset="utf-8"><title>札予測詳細</title>'
       f'<style>{css}</style></head><body><main><header><h1>競技かるた｜札予測・詳細結果</h1>'
       f'<div class="meta">実行ID: {fmt(RUN_ID)} ／ 入力: {fmt(source)}<br>'
       f'読手: {fmt(reader)} ／ モード: {fmt(mode)} ／ 辞書: {fmt(dic.name)}</div></header>'
       f'<div class="summary">最終予測：{fmt(final_id)}番 ／ 類似度 {final_pct:.1f}% ／ '
       f'認識時間 {session.elapsed:.3f} 秒 ／ イベント数 {len(session.detail_rows)}</div>'
       + ''.join(blocks) + '<p class="note">加点・満点は重み付けと漢字換算を反映。緑色は各時点の第1候補。CSVには上位5件だけでなく全候補を保存します。</p>'
       '</main></body></html>')
    report.write_text(html_text, encoding='utf-8')
    print(f'[見やすい詳細HTML] {report}')
    return dest


def save_rms_plot(session: RecognitionSession, basename: str) -> Path | None:
    """元のWAV解析に準じたRMS画像。マイク・WAV両方で保存する。"""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print('[RMS] matplotlibがないためグラフをスキップしました。')
        return None
    if not session.rms_data:
        return None
    dest_dir = RUN_PLOT_DIR if RUN_PLOT_DIR is not None else PLOT_DIR
    dest_dir.mkdir(parents=True, exist_ok=True)
    times, volumes = zip(*session.rms_data)
    fig, ax = plt.subplots(figsize=(12, 4.5))
    ax.plot(times, volumes, color='#1f77b4', lw=1.2, alpha=.85,
            label='RMS Volume')
    ax.axhline(VOLUME_THRESHOLD, color='red', ls='--', lw=1,
               label=f'Threshold ({VOLUME_THRESHOLD})')
    if session.upper_start is not None:
        ax.axvline(session.upper_start, color='green', ls='-.', lw=1.5,
                   label=f'Uonoku Start ({session.upper_start:.2f}s)')
    if session.upper_start is not None and session.end_at is not None:
        deadline = session.upper_start + session.end_at
        if deadline <= max(times):
            ax.axvline(deadline, color='purple', ls='--', lw=1,
                       label=f'End Threshold ({session.end_at:.2f}s)')
    for marker_t, marker_name in getattr(session, 'phase_markers', []):
        ax.axvline(marker_t, ls='--', lw=1, color='gray', alpha=.8,
                   label=f'{marker_name} ({marker_t:.2f}s)')
    max_rms = max(max(volumes), VOLUME_THRESHOLD * 2, 0.001)
    for i, (t, word) in enumerate(session.word_markers):
        ax.axvline(t, color='darkorange', ls=':', alpha=.55)
        ax.text(t, max_rms * (0.68 if i % 2 else 0.85), f' {word}',
                rotation=90, va='bottom', color='darkorange', fontsize=8)
    cid, _ = session.result()
    ax.set(title=f'RMS Analysis: {basename} (ID: {cid})',
           xlabel='Time (seconds)', ylabel='RMS Volume')
    ax.set_ylim(bottom=0, top=max_rms * 1.35)
    ax.grid(True, ls=':', alpha=.6)
    ax.legend(loc='upper right')
    fig.tight_layout()
    safe_name = make_short_output_stem(basename)
    output = dest_dir / f'{safe_name}_rms.png'
    fig.savefig(output, dpi=120)
    plt.close(fig)
    return output


def process_wav(path: Path, model: Model, dic: DictionaryData, predictor: CardPredictor,
                reader: str, gender: str, limit: float, start_mode: str, mode_label: str, correction: float):
    try:
        with wave.open(str(path), 'rb') as wf:
            if (wf.getnchannels(), wf.getsampwidth(), wf.getframerate(), wf.getcomptype()) != (1, 2, SAMPLE_RATE, 'NONE'):
                print(f'[スキップ] {path.name}: PCM 16bit/16kHz/モノラルWAVが必要です。')
                return
            session = RecognitionSession(create_recognizer(model, dic), predictor, limit, start_mode, False, correction)
            last_upper_voice = None
            upper_end_found = False
            while True:
                data = wf.readframes(CHUNK_SIZE)
                if not data:
                    break
                values = np.frombuffer(data, dtype=np.int16).astype(np.float32)
                dt = len(values) / SAMPLE_RATE
                volume = float(np.sqrt(np.mean(values * values)) / 32768.0) if len(values) else 0.0
                if not session.finished:
                    session.consume(data)
                    if session.upper_start is not None and volume >= VOLUME_THRESHOLD:
                        last_upper_voice = session.total_time
                else:
                    # 判定を止めても、上の句が終了するまでRMS取得を継続。
                    session.total_time += dt
                    session.last_rms = volume
                    session.rms_data.append((session.total_time, volume))
                    if volume >= VOLUME_THRESHOLD:
                        last_upper_voice = session.total_time
                    if (session.elapsed >= UPPER_MIN_TOTAL_SECONDS
                            and last_upper_voice is not None
                            and session.total_time-last_upper_voice >= UPPER_END_SILENCE_SECONDS):
                        session.phase_markers.append((session.total_time, 'Upper Phrase End'))
                        upper_end_found = True
                        break
            if not session.finished:
                session.finish()
            if not upper_end_found:
                print('[注意] WAV内で上の句終了を確認できませんでした。RMS画像は途中までです。')
    except (OSError, EOFError, wave.Error) as exc:
        print(f'[エラー] {path}: {exc}')
        return
    match = re.search(r'set_(\d+)', path.name)
    expected = card_id(match.group(1)) + '_u' if match else '-'
    save_result(reader, gender, mode_label, dic, expected, path.name, session)
    detail_path = save_detail_csv(session, path.name, reader, gender, mode_label, dic, expected)
    print(f'[詳細CSV] {detail_path}')
    print(f'[WAV保存] {archive_input_wav(path)}')
    plot_path = save_rms_plot(session, path.stem)
    if plot_path:
        print(f'[RMS画像] {plot_path}')


def run_microphone(model: Model, dic: DictionaryData, predictor: CardPredictor,
                   reader: str, gender: str, limit: float, start_mode: str,
                   mic_mode: str, device_id: int | None, correction: float):
    """各RMS画像を「前札下の句 -> 間 -> 上の句認識 -> 上の句残り」に統一。

    RMS/録音を保存する時点は「上の句が終わったと判定したとき」。
    次に流れる下の句は次の札の先頭データとして引き継ぐ。
    """
    try:
        import sounddevice as sd
    except ImportError as exc:
        raise RuntimeError('マイク入力には sounddevice が必要です: pip install sounddevice') from exc
    import queue
    audio_queue = queue.Queue(maxsize=120)
    lost_chunks = [0]

    def callback(indata, frames, callback_time, status):
        try:
            audio_queue.put_nowait(bytes(indata))
        except queue.Full:
            lost_chunks[0] += 1

    session = RecognitionSession(create_recognizer(model, dic), predictor, limit,
                                 start_mode, correction=correction, notebook_display=True)
    mode_label = 'マイク単発' if mic_mode == '1' else 'マイク連続'
    count = 0
    audio_chunks = []  # 現在の画像に対応する音声: 前札下の句から上の句終了まで
    recording_name = None
    gate = None  # None=認識処理 / upper_tail / lower_wait / lower_active
    last_speech_at = None
    lower_chunks = []  # 次の札に引き継ぐ「前札下の句」
    lower_rms = []
    lower_elapsed = 0.0
    lower_last_voice = None
    last_wait_render = -1.0
    final_plot_saved = False

    print('Ctrl+C で終了。マイク: 16kHz / モノラル / 16bit')
    try:
        with sd.RawInputStream(samplerate=SAMPLE_RATE, blocksize=CHUNK_SIZE,
                               device=device_id, channels=1, dtype='int16', callback=callback):
            while True:
                try:
                    data = audio_queue.get(timeout=0.5)
                except queue.Empty:
                    continue
                samples = np.frombuffer(data, dtype=np.int16).astype(np.float32)
                dt = len(samples) / SAMPLE_RATE
                rms = float(np.sqrt(np.mean(samples * samples)) / 32768.0) if len(samples) else 0.0
                speaking = rms >= VOLUME_THRESHOLD

                if gate is None:
                    audio_chunks.append(data)
                    session.consume(data)
                    if not session.finished:
                        continue

                    # 認識結果だけを先に確定。画像は「上の句終了」の後に保存する。
                    count += 1
                    recording_name = f'microphone_{datetime.now().strftime("%Y%m%d_%H%M%S_%f")}_{count:03d}'
                    save_result(reader, gender, mode_label, dic, '-', recording_name, session)
                    detail_path = save_detail_csv(session, recording_name, reader, gender, mode_label, dic)
                    print(f'[結果保存] {RESULTS_CSV} / 詳細: {detail_path}')
                    gate = 'upper_tail'
                    session.phase = 'upper_tail'
                    last_speech_at = session.total_time if speaking else session.last_voice_time
                    session.phase_markers.append((session.total_time, 'Recognition End'))
                    session.show_status()
                    continue

                if gate == 'upper_tail':
                    # 上の句の残りが続く間もRMSと音声を記録する。
                    audio_chunks.append(data)
                    session.total_time += dt
                    session.last_rms = rms
                    session.rms_data.append((session.total_time, rms))
                    if speaking:
                        last_speech_at = session.total_time
                    if (session.elapsed >= UPPER_MIN_TOTAL_SECONDS
                            and last_speech_at is not None
                            and session.total_time - last_speech_at >= UPPER_END_SILENCE_SECONDS):
                        session.phase_markers.append((session.total_time, 'Upper Phrase End'))
                        # 下の句がまだ入っていない段階で保存する。
                        wav_path = save_audio_wav(audio_chunks, recording_name)
                        rms_path = save_rms_plot(session, recording_name)
                        print(f'[上の句終了] 録音: {wav_path} / RMS: {rms_path}')
                        audio_chunks = []
                        final_plot_saved = True
                        if mic_mode == '1':
                            break
                        gate = 'lower_wait'
                        session.phase = 'lower_wait'
                        lower_chunks = []
                        lower_rms = []
                        lower_elapsed = 0.0
                        lower_last_voice = None
                        session.show_status()
                        continue

                elif gate == 'lower_wait':
                    session.total_time += dt
                    session.last_rms = rms
                    # 上の句後の間は、今回のRMS画像には含めない。
                    if speaking:
                        gate = 'lower_active'
                        session.phase = 'lower_active'
                        lower_chunks = [data]
                        lower_elapsed = dt
                        lower_rms = [(lower_elapsed, rms)]
                        lower_last_voice = lower_elapsed
                        session.show_status()
                    continue

                elif gate == 'lower_active':
                    session.total_time += dt
                    # この下の句を「次の札の前札下の句」に引き継ぐ。
                    lower_chunks.append(data)
                    lower_elapsed += dt
                    lower_rms.append((lower_elapsed, rms))
                    session.last_rms = rms
                    if speaking:
                        lower_last_voice = lower_elapsed
                    if (lower_last_voice is not None
                            and lower_elapsed - lower_last_voice >= WAIT_SECONDS):
                        # 下の句＋直後の無音を次セットの先頭に持ち越す。
                        session.reset()
                        session.phase = 'upper_wait'
                        session.total_time = lower_elapsed
                        session.rms_data = lower_rms.copy()
                        session.phase_markers = [(0.0, 'Previous Lower Phrase Start'),
                                                 (lower_elapsed, 'Previous Lower Phrase End')]
                        session.last_rms = rms
                        audio_chunks = lower_chunks.copy()
                        lower_chunks = []
                        lower_rms = []
                        lower_elapsed = 0.0
                        lower_last_voice = None
                        recording_name = None
                        gate = None
                        final_plot_saved = False
                        session.show_status('前札下の句終了・次の上の句待機')
                        continue

                if session.total_time - last_wait_render >= 0.2:
                    session.show_status()
                    last_wait_render = session.total_time

    except KeyboardInterrupt:
        print('\n[終了] マイク認識を停止しました。')
    finally:
        # 上の句終了前の手動中断などは、完成画像と混同しないよう _partial にする。
        if gate == 'upper_tail' and recording_name and not final_plot_saved:
            try:
                rms_path = save_rms_plot(session, recording_name + '_partial')
                print(f'[途中終了・不完全なRMS画像] {rms_path}')
            except Exception as exc:
                print(f'[RMS保存失敗] {exc}')
        if audio_chunks:
            try:
                partial_name = recording_name or f'microphone_interrupted_{datetime.now().strftime("%Y%m%d_%H%M%S_%f")}'
                wav_path = save_audio_wav(audio_chunks, partial_name, partial=True)
                print(f'[途中終了時の録音WAV] {wav_path}')
            except Exception as exc:
                print(f'[録音WAV保存失敗] {exc}')
        if lost_chunks[0]:
            print(f'[注意] 音声チャンクの欠落: {lost_chunks[0]}回（処理負荷を確認）')


In [5]:
def main():
    print('=== 競技かるた Vosk 音声認識・札予測 ===')
    model_choice = input('【Model】1: Standard / 2: Small -> ').strip()
    model_path = ROOT / ('model_standard' if model_choice == '1' else 'model_small')
    if not model_path.is_dir():
        print(f'[エラー] Voskモデルが見つかりません: {model_path}')
        return

    reader, gender = reader_prompt()
    print('\n【辞書】')
    for k, (name, path, _) in DICTIONARIES.items():
        print(f' {k}: {name}')
    choice = input('選択 [Enter: 7] -> ').strip() or '7'
    if choice not in DICTIONARIES:
        choice = '7'
    new_mode = '1'
    if choice == '7':
        new_mode = input('【使用語彙】1: 全パーツ / 2: VOSK特化2列のみ [Enter: 1] -> ').strip() or '1'

    source_choice = input('【入力】1: WAVファイル / 2: リアルタイムマイク -> ').strip()
    source_choice = source_choice if source_choice in ('1', '2') else '2'
    custom_id = 'CUSTOM'
    if choice == '6':
        raw = input('【登録札番号】(例: 1 または 001) -> ').strip()
        custom_id = card_id(raw) if raw else 'CUSTOM'
    dictionary = load_dictionary(choice, custom_id, new_mode)
    export_dictionary_csv(dictionary)
    predictor = CardPredictor(dictionary)

    start_mode = input('【認識開始】1: 上の句から直接 / 2: 前札下の句終了後 [Enter: 1] -> ').strip() or '1'
    correction_text = input(f'【認識時間の補正値(秒)】[Enter: {TIME_CORRECTION_SECONDS}] -> ').strip()
    try:
        correction = max(0.0, float(correction_text)) if correction_text else TIME_CORRECTION_SECONDS
    except ValueError:
        correction = TIME_CORRECTION_SECONDS
    limit = input_number('【初語が見つからない場合の安全終了時間(秒)】', DEFAULT_LIMIT_SECONDS)
    print(f'\nVoskモデル読み込み中: {model_path}')
    SetLogLevel(-1)
    model = Model(str(model_path))
    start_run()  # main()の実行ごとに新しい詳細CSVフォルダと実行IDを生成

    if source_choice == '2':
        mode = input('【マイクモード】1: 単発 / 2: 連続 [Enter: 2] -> ').strip() or '2'
        device_text = input('【入力デバイス番号】Enter: OS標準 / l: 一覧表示 -> ').strip().lower()
        if device_text == 'l':
            import sounddevice as sd
            print(sd.query_devices())
            device_text = input('【入力デバイス番号】Enter: OS標準 -> ').strip()
        device_id = int(device_text) if device_text.isdecimal() else None
        run_microphone(model, dictionary, predictor, reader, gender, limit,
                       start_mode, mode, device_id, correction)
        return

    mode = input('【WAVモード】1: 単発 / 2: 連続 / 3: ランダム [Enter: 2] -> ').strip() or '2'
    if mode == '1':
        raw = input('【上の句番号】(例: 1 または 001) -> ').strip()
        cid = card_id(raw)
        files = sorted(RECORD_DIR.glob(f'set_{cid}_*.wav'))
        if not files:
            files = sorted(RECORD_DIR.glob(f'set_{cid}.wav'))
        mode_label = '単発WAVテスト'
    else:
        files = sorted(RECORD_DIR.glob('set_*.wav'))
        if mode == '3':
            random.shuffle(files)
        mode_label = 'ランダムWAVテスト' if mode == '3' else '連続WAVテスト'
    if not files:
        print(f'[エラー] 対象WAVがありません: {RECORD_DIR}')
        return
    print(f'\n対象: {len(files)} 件')
    for i, path in enumerate(files, 1):
        print(f'\n[{i}/{len(files)}] {path.name}')
        process_wav(path, model, dictionary, predictor, reader, gender,
                    limit, start_mode, mode_label, correction)
    print(f'\n[解析完了] 結果: {RESULTS_CSV}')
    print(f'[RMSグラフ] {RUN_PLOT_DIR}')
    print(f'[音声ファイル] {RUN_AUDIO_DIR}')

In [15]:
dictionary = load_dictionary("7", new_mode="1")
export_dictionary_csv(dictionary)

predictor = CardPredictor(dictionary)

読み込み列: カタカナ決まり字 / ひらがな決まり字 / カタカナ決まり字パーツ / ひらがな決まり字パーツ / カタカナパーツ(1句目) / ひらがなパーツ(1句目) / カタカナパーツ(2句目) / ひらがなパーツ(2句目) / 手動追加パーツ / 誤識別追加 / VOSK特化辞書カタカナ / VOSK特化辞書ひらがな
[辞書ロード完了] 新・かるた統合辞書: 100札 / 1183語 / 2828文字（全札の重複を除外）
[認識時間] CSVの数値は札ごとの最短認識時間です（指定札数: 100）。
[重み付け] 列=True / 単語=True / 札＋単語=True
[辞書内容CSV保存] C:\Users\misak\Desktop\karuta_vosk\results\karuta_dictionary_preview.csv


In [18]:
main()

=== 競技かるた Vosk 音声認識・札予測 ===

【辞書】
 1: ひらがな版（統一文字数辞書）
 2: カタカナ版（統一文字数辞書）
 3: 決まり字・手動追加版（サマリー辞書）
 4: VOSK専用辞書
 5: VOSK特化辞書（旧形式）
 6: カスタムテキスト入力（動的辞書）
 7: 新・かるた統合辞書
読み込み列: カタカナ決まり字 / ひらがな決まり字 / カタカナ決まり字パーツ / ひらがな決まり字パーツ / カタカナパーツ(1句目) / ひらがなパーツ(1句目) / カタカナパーツ(2句目) / ひらがなパーツ(2句目) / 手動追加パーツ / 誤識別追加 / VOSK特化辞書カタカナ / VOSK特化辞書ひらがな
[辞書ロード完了] 新・かるた統合辞書: 100札 / 1185語 / 2832文字（全札の重複を除外）
[認識時間] CSVの数値は札ごとの最短認識時間です（指定札数: 100）。
[重み付け] 列=True / 単語=True / 札＋単語=True
[辞書内容CSV保存] C:\Users\misak\Desktop\karuta_vosk\results\karuta_dictionary_preview.csv

Voskモデル読み込み中: c:\Users\misak\Desktop\karuta_vosk\model_small
[今回の実行ID] 20261114_182133_904169
[詳細CSV保存先] c:\Users\misak\Desktop\karuta_vosk\recognition_details\20261114_182133_904169
[RMS画像保存先] c:\Users\misak\Desktop\karuta_vosk\rms_plots\20261114_182133_904169
[録音WAV保存先] c:\Users\misak\Desktop\karuta_vosk\recordings\20261114_182133_904169


HTML(value='<div style="box-sizing:border-box;max-width:780px;padding:5px 9px;\n             border:1px solid …

Ctrl+C で終了。マイク: 16kHz / モノラル / 16bit
[見やすい詳細HTML] c:\Users\misak\Desktop\karuta_vosk\recognition_details\20261114_182133_904169\001_details20261114_182153_708976.html
[結果保存] c:\Users\misak\Desktop\karuta_vosk\results\batch_recognition_history.csv / 詳細: c:\Users\misak\Desktop\karuta_vosk\recognition_details\20261114_182133_904169\001_details20261114_182153_708976.csv


C:\Users\misak\AppData\Local\Temp\ipykernel_29308\3681192386.py:257: UserWarning: Glyph 31179 (\N{CJK UNIFIED IDEOGRAPH-79CB}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_29308\3681192386.py:257: UserWarning: Glyph 37326 (\N{CJK UNIFIED IDEOGRAPH-91CE}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_29308\3681192386.py:257: UserWarning: Glyph 12383 (\N{HIRAGANA LETTER TA}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_29308\3681192386.py:257: UserWarning: Glyph 12398 (\N{HIRAGANA LETTER NO}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_29308\3681192386.py:257: UserWarning: Glyph 26126 (\N{CJK UNIFIED IDEOGRAPH-660E}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_29308\3681192386.py:257: UserWarning: Glyph 20035 (\N{CJK UNIF

[上の句終了] 録音: c:\Users\misak\Desktop\karuta_vosk\recordings\20261114_182133_904169\001_20261114_182153_708976_audio.wav / RMS: c:\Users\misak\Desktop\karuta_vosk\rms_plots\20261114_182133_904169\001_20261114_182153_708976_rms.png
[見やすい詳細HTML] c:\Users\misak\Desktop\karuta_vosk\recognition_details\20261114_182133_904169\002_details20261114_182221_818234.html
[結果保存] c:\Users\misak\Desktop\karuta_vosk\results\batch_recognition_history.csv / 詳細: c:\Users\misak\Desktop\karuta_vosk\recognition_details\20261114_182133_904169\002_details20261114_182221_818234.csv


C:\Users\misak\AppData\Local\Temp\ipykernel_29308\3681192386.py:257: UserWarning: Glyph 12450 (\N{KATAKANA LETTER A}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_29308\3681192386.py:257: UserWarning: Glyph 12522 (\N{KATAKANA LETTER RI}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_29308\3681192386.py:257: UserWarning: Glyph 36942 (\N{CJK UNIFIED IDEOGRAPH-904E}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_29308\3681192386.py:257: UserWarning: Glyph 12366 (\N{HIRAGANA LETTER GI}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_29308\3681192386.py:257: UserWarning: Glyph 12377 (\N{HIRAGANA LETTER SU}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_29308\3681192386.py:257: UserWarning: Glyph 12390 (\N{HIRAGANA LETTER TE}) miss

... (93 outputs left out)

C:\Users\misak\AppData\Local\Temp\ipykernel_29308\3681192386.py:257: UserWarning: Glyph 12374 (\N{HIRAGANA LETTER ZA}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_29308\3681192386.py:260: UserWarning: Glyph 12374 (\N{HIRAGANA LETTER ZA}) missing from font(s) DejaVu Sans.
  fig.savefig(output, dpi=120)


[上の句終了] 録音: c:\Users\misak\Desktop\karuta_vosk\recordings\20261114_182133_904169\095_20261114_185845_727808_audio.wav / RMS: c:\Users\misak\Desktop\karuta_vosk\rms_plots\20261114_182133_904169\095_20261114_185845_727808_rms.png
[見やすい詳細HTML] c:\Users\misak\Desktop\karuta_vosk\recognition_details\20261114_182133_904169\096_details20261114_185910_120306.html
[結果保存] c:\Users\misak\Desktop\karuta_vosk\results\batch_recognition_history.csv / 詳細: c:\Users\misak\Desktop\karuta_vosk\recognition_details\20261114_182133_904169\096_details20261114_185910_120306.csv
[上の句終了] 録音: c:\Users\misak\Desktop\karuta_vosk\recordings\20261114_182133_904169\096_20261114_185910_120306_audio.wav / RMS: c:\Users\misak\Desktop\karuta_vosk\rms_plots\20261114_182133_904169\096_20261114_185910_120306_rms.png
[見やすい詳細HTML] c:\Users\misak\Desktop\karuta_vosk\recognition_details\20261114_182133_904169\097_details20261114_185934_732678.html
[結果保存] c:\Users\misak\Desktop\karuta_vosk\results\batch_recognition_history.csv / 

C:\Users\misak\AppData\Local\Temp\ipykernel_29308\3681192386.py:257: UserWarning: Glyph 12371 (\N{HIRAGANA LETTER KO}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_29308\3681192386.py:257: UserWarning: Glyph 25551 (\N{CJK UNIFIED IDEOGRAPH-63CF}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\misak\AppData\Local\Temp\ipykernel_29308\3681192386.py:260: UserWarning: Glyph 12371 (\N{HIRAGANA LETTER KO}) missing from font(s) DejaVu Sans.
  fig.savefig(output, dpi=120)
C:\Users\misak\AppData\Local\Temp\ipykernel_29308\3681192386.py:260: UserWarning: Glyph 25551 (\N{CJK UNIFIED IDEOGRAPH-63CF}) missing from font(s) DejaVu Sans.
  fig.savefig(output, dpi=120)


[上の句終了] 録音: c:\Users\misak\Desktop\karuta_vosk\recordings\20261114_182133_904169\098_20261114_185957_325211_audio.wav / RMS: c:\Users\misak\Desktop\karuta_vosk\rms_plots\20261114_182133_904169\098_20261114_185957_325211_rms.png
[見やすい詳細HTML] c:\Users\misak\Desktop\karuta_vosk\recognition_details\20261114_182133_904169\099_details20261114_190022_730021.html
[結果保存] c:\Users\misak\Desktop\karuta_vosk\results\batch_recognition_history.csv / 詳細: c:\Users\misak\Desktop\karuta_vosk\recognition_details\20261114_182133_904169\099_details20261114_190022_730021.csv
[上の句終了] 録音: c:\Users\misak\Desktop\karuta_vosk\recordings\20261114_182133_904169\099_20261114_190022_730021_audio.wav / RMS: c:\Users\misak\Desktop\karuta_vosk\rms_plots\20261114_182133_904169\099_20261114_190022_730021_rms.png
[見やすい詳細HTML] c:\Users\misak\Desktop\karuta_vosk\recognition_details\20261114_182133_904169\100_details20261114_190046_139174.html
[結果保存] c:\Users\misak\Desktop\karuta_vosk\results\batch_recognition_history.csv / 